In [1]:
import copy
import numpy as np
from dataclasses import dataclass, field, fields
import matplotlib.pyplot as plt
from typing import Callable, Dict, Tuple, Optional
from scipy.optimize import fsolve
from scipy.special import lambertw
from scipy.special import erfc, erfcinv
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
ignores = np.seterr(all='ignore')

In [2]:
class EnergyDefaults:
    # Hardware defaults
    hardware = {
        'f_mcu': 16e6, 'f_s': 1e3, 'voltage': 3.3,
        'I_mcu': 5.0e-3, 'I_adc': 2.0e-3, 'I_act': 3.0e-3, 'I_ext': 1.0e-3,
        'I_sleep': 0.01e-3, 'I_wake': 5.0e-3,
    }
    # Networking defaults
    communication = {
        'bit_rate_up_ir': 100e3, 'bit_rate_up_rf': 250e3, 'bit_rate_dw': 1e6,
        't_init': 5e-3, 't_wait': 10e-3,
        "T_cycle":1,
        'harvesting_hours': 5
    
    }
    # Computational/Sensing task defaults
    tasks = {
        'L_up_bits': 1024, 'L_dw_bits': 64,
        'N_s_up': 100, 'N_c_up': 1e5,
        'N_s_dw': 0, 
    }
    battery = {
        'battery_capacity_mAh': 500,
        'initial_soc': 1,
        'V_batt': 3.6,
        
    }

In [3]:
#Constants
class Constants:
    """Physical constants used across the simulation."""
    q = 1.60217663e-19       # Elementary charge
    kB = 1.380649e-23        # Boltzmann constant
    c0 = 299792458.0         # Speed of light
    hP = 6.62607015e-34      # Planck constant
    T0 = 300.0               # Standard Temperature (K)
    bK = 2.8977729e-3        # Wien Constant
    pd_peak = 2e9
    T = 298                  # Temperature
    eo = 8.854e-12
    # Vectors
    zp = np.array([0, 0, 1])
    zm = np.array([0, 0, -1])
    xp = np.array([1, 0, 0])
    xm = np.array([-1, 0, 0])
    yp = np.array([0, 1, 0])
    ym = np.array([0, -1, 0])

@dataclass
class DefaultSimValues:
    fov = np.pi/2
    m = 1
    A = 1e-4
    n = Constants.zp
    sensitivity = -100 #dBm

In [64]:
DUMMY_DESIGN = {
    'meta': {
        'name': 'Hybrid_VLC_RF_IoT_Network_Daily',
        'description': 'Daily Energy Budget with RIS and Windows.'
    },

    'environment': {
        'dimensions': np.array([5.0, 5.0, 3.0]),
        'wall_resolution': (20,20),
        'reflectivity': {'floor': 0.3, 'ceiling': 0.8, 'walls': 0.5},

        # (RIS/Windows)
        'special_surfaces': [
            {
                'type': 'RIS', 'name': 'ris_west',
                'center': [5, 2.5, 1.5], 'dims': [1.0, 1.0],
                'const_axis': 0, 'normal': [1,0,0],
                'resolution': (2,2), 'mode': 10.0, 'reflectivity': 0.9
            },

            {
                'type': 'RIS', 'name': 'ris_east',
                'center': [0, 2.5, 1.5], 'dims': [1.0, 1.0],
                'const_axis': 0, 'normal': [1,0,0],
                'resolution': (2,2), 'mode': 10.0, 'reflectivity': 0.9
            },

            {
                'type': 'window', 'name': 'win_north',
                'center': [2.5, 5, 1.5], 'dims': [2.0, 1.0],
                'const_axis': 1, 'normal': [0,-1,0],
                'resolution': (2,2), 'reflectivity': 0.1
            }
        ],


    },

    'nodes': {
        'masters': {
            'positions': np.array([[2.5, 2.5, 3.0], [3,3,3]]),
            'nT':   Constants.zm,
            'nR':   Constants.zm,
            'm':      1,
            'tx_power':  1,
            'rx_area':   1e-4,
            'IR_pass_filter': True,
        },
        'sensors': {
            'positions': np.array([
                [1.0, 1.0, 0.85], [4.0, 4.0, 0.85],
                [2.5, 2.5, 0.85], [2,2,0]
            ]),
            'rx_area':    np.full(4, 1e-4),
            'nT': np.repeat(Constants.zp[None, :], 4, axis=0),
            'nR': np.repeat(Constants.zp[None, :], 4, axis=0),
            # 0=PD, 1=SP
            'rx_type':     np.array([1, 0, 1,0]),
            #for PVs filters are ignored
            'VLC_pass_filter': np.array([False,False,False,True]),
            # 1=RF, 0=IR
            'uplink_type': np.array([0, 0, 0,1]),
            "IR_tx_power": 0.015,
            "RF_tx_power": -20,
            # Energy Storage
            
        },
        'ambient_nodes': {
            'positions': np.array([[3, 3, 3.0]]),
            'nT':   np.array([[0, 0, -1]]),
            'tx_power':    2,
            'm' : 1
        }
    },

    'TIA': {
        'RF': 1e6, 'CF': 1e-9,
        'Vn': 15e-9, 'In': 400e-15,
        'fncV': 1e3, 'fncI': 1e3,
        'temperature': 300
    },

    'PV': {
        'Rs': 1, 'Rsh': 1000, 'Jsc': 35e-3, 'Voc': 0.64, 'n': 1.6

    },

    'MPP': {
        'efficiency' : 0.8
    },

    'energy_profile': {
        'hardware': {
            'f_mcu':   np.array([80e6, 80e6, 16e6, 48e6]),     # Clock Speed (Hz)
            'f_s':     np.array([1e3, 10e3, 1e3, 5e3]),        # Sampling Freq (Hz)
            'voltage': np.array([3.3, 3.3, 3.3, 1.8]),         # Operating Voltage (V)
            'I_mcu':   np.array([4.0, 12.0, 4.0, 8.0])*1e-3,   # Active MCU (mA)
            'I_adc':   2.0*1e-3,                               # ADC Current (mA)
            'I_act':   3.0*1e-3,                               # RX Front-end (mA)
            'I_ext':   np.array([0.5, 5.0, 1.0, 2.0])*1e-3,    # Ext. Sensors (mA)
            'I_sleep': 0.01*1e-3,                              # Sleep Current (mA)
            'I_wake':  5.0*1e-3,                               # Wake-up Current (mA)
        },
        'tasks': {
            'N_samples_up': np.array([100, 5000, 100, 500]), # Samples for UL
            'N_cycles_up':  np.array([1e5, 5e6, 1e5, 8e5]),  # Processing Cycles for UL
            'L_up_bits':    np.array([1024, 8192, 512, 2048]),# Payload size
            'L_down_bits':  64,                               # ACK size
           
        }
    ,
        'communication': {
            'Rb_up': 100e3, # bps
            'Rb_down':  15e3,  # bps (VLC)
            'n_sp_d': 0.4, #spectral efficiency downlink
            'n_sp_u':0.4, #spectal efficiency uplink
            't_init': 5e-3,          # Boot time (s)
            #'t_wait': np.array([10e-3, 10e-3, 10e-3, 20e-3]), # ACK wait (s)
        },
        'battery': {
            'battery_capacity_mAh': 500,
            'initial_soc':     1,
            'V_batt': 3.6
        }
    },

    'protocol': {
        'T_cycle': 6,
        #'harvesting_hours': 8
    }
}

In [65]:
@dataclass
class Elements:
    """
    Base data structure representing a batch of spatial elements in 3D space.
    
    This class handles the position and orientation vectors for N elements and 
    provides methods to merge multiple batches together.
    
    Attributes:
        r (np.ndarray): Position vectors of shape (N, 3).
        n (np.ndarray): Orientation/Normal vectors of shape (N, 3). Defaults to [0, 0, 1].
    """
    r: np.ndarray  # (N, 3) Position
    n: np.ndarray  = field(default_factory=lambda: np.array([0,0,1]))  # (N, 3) Orientation/Normal


    def __post_init__(self):
        """
        Validates inputs and ensures data is stored as (N, 3) numpy arrays.
        
        Calculates the number of elements (N) based on the position array 'r'.
        """
        # Cast to numpy arrays first
        self.r = np.array(self.r)
        self.n = np.array(self.n)

        if self.r.ndim == 1:
            self.r = self.r.reshape(1, 3)

        self.N = self.r.shape[0]
        self.n = to_vec_Nx3(self.N, self.n)
        norms = np.linalg.norm(self.n, axis=1, keepdims=True)
        if not np.allclose(norms, 1.0):
            self.n = self.n / norms

    def __add__(self, other):
        """
        Allows using '+' to combine two batches of elements:
        batch_combined = batch1 + batch2
        
        Args:
            other (Elements): The other batch to append to this one.
            
        Returns:
            Elements: A new instance containing the vertically stacked data of both batches.
            
        Raises:
            TypeError: If 'other' is not the same class type.
            ValueError: If optional fields are present in one object but not the other.
        """
        if other is None:
            return copy.deepcopy(self)

        # Ensure we are adding compatible types (e.g., Rx + Rx, not Rx + Tx)
        if not isinstance(other, type(self)):
            raise TypeError(f"Cannot add {type(other)} to {type(self)}")

        # Create a copy to store the result
        new_obj = copy.deepcopy(self)

        # Automatically loop through all defined fields (r, n, A, + subclass fields)
        for field in fields(self):
            name = field.name

            val_self = getattr(self, name)
            val_other = getattr(other, name)

            # Handle optional fields that might be None (like refl)
            if val_self is None and val_other is None:
                continue
            if val_self is None or val_other is None:
                # If one is defined and the other isn't, we can't safely stack
                raise ValueError(f"Field '{name}' is None in one object but not the other.")

            # Stack the arrays vertically
            stacked_val = np.vstack([val_self, val_other])
            setattr(new_obj, name, stacked_val)

        # Manually update N (since it's not a dataclass field, the loop skips it)
        new_obj.N = new_obj.r.shape[0]

        return new_obj

    @classmethod
    def merge(cls, batch_list):
        """
        Merges a list of Element batches into one.
        
        This is more efficient than repeated addition for large lists. It also handles
        optional fields (like 'refl') by padding missing values with zeros if necessary.
        
        Args:
            batch_list (list): A list of Element objects (or subclasses) to merge.
            
        Returns:
            Elements: A single merged instance, or None if the list is empty.
        """
        if not batch_list:
            return None

        # Use the first item as a template
        merged = copy.deepcopy(batch_list[0])

        # Loop through fields and stack everything
        for field in fields(cls):
            name = field.name


            values = [getattr(b, name) for b in batch_list]

            # Handle cases where some batches might have None for optional fields
            if any(v is None for v in values):
                # If all are None, skip (result stays None)
                if all(v is None for v in values):
                    continue

                # Otherwise, fill None gaps with Zeros so we don't crash
                # We use 'b.N' to know how many zeros to generate for that batch
                valid_sample = next(v for v in values if v is not None)
                cols = valid_sample.shape[1] if valid_sample.ndim > 1 else 1

                values = [
                    v if v is not None else np.zeros((b.N, cols))
                    for v, b in zip(values, batch_list)
                ]

            
            setattr(merged, name, np.vstack(values))

        # Update the total count N
        merged.N = merged.r.shape[0]

        return merged

@dataclass
class OpticalTxElements(Elements):
    """
    Properties specific to Optical Transmitters (e.g., LEDs, Lasers).
    
    Attributes:
        p (np.ndarray): Optical power per element (Watts).
        m (np.ndarray): Lambertian order of emission (mode number).
    """
    """Properties specific to Transmitters."""
    p: np.ndarray = 1
    m: np.ndarray = DefaultSimValues.m

    def __post_init__(self):
        super().__post_init__() # This creates self.N

        self.p = np.array(self.p)
        self.m = np.array(self.m)

        self.p = to_scal_Nx1(self.N, self.p)
        self.m = to_scal_Nx1(self.N, self.m)

@dataclass
class OpticalRxElements(Elements):
    """
    Properties specific to Optical Receivers.
    
    Attributes:
        A (np.ndarray): Active detection area (m^2).
        fov (np.ndarray): Field of View (half-angle in radians).
        refl (np.ndarray): Reflection coefficient (optional).
        type_Rx (np.ndarray): Identifier for receiver type (e.g., 0 for photodiode / surface elements, 1 for photovoltaic panel).
    """
    """Properties specific to Receivers."""
    A: np.ndarray = DefaultSimValues.A
    fov: np.ndarray = DefaultSimValues.fov
    refl: np.ndarray = None
    type_Rx: np.ndarray = 0

    def __post_init__(self):
        super().__post_init__()

        self.fov = np.array(self.fov)
        self.fov = to_scal_Nx1(self.N, self.fov)
        self.type_Rx = to_scal_Nx1(self.N, self.type_Rx)
        self.A = to_scal_Nx1(self.N,self.A)

        if self.refl is not None:
            self.refl = np.array(self.refl)
            self.refl = to_scal_Nx1(self.N, self.refl)

@dataclass
class RFTxElements(Elements):
    """
    Properties specific to RF (Radio Frequency) Transmitters.
    
    Attributes:
        p (np.ndarray): Transmission power (dBm).
    """
    """Properties specific to Receivers."""
    p : np.ndarray = None #dBm


    def __post_init__(self):
        super().__post_init__()

        self.p = np.array(self.p)

In [66]:
class SpectralPhysics:
    """
    Physics engine for calculating effective responsivity by integrating spectral 
    overlaps of light sources, detectors, and optical filters.
    
    This class manages spectral data generation (using Gaussian approximations 
    or blackbody radiation), polynomial fitting for detector sensitivity, and 
    numerical integration to determine system efficiency.
    """
    # 1. Store constants clearly to avoid magic numbers
    L_MIN, L_MAX = 300e-9, 1200e-9
    GRID_POINTS = 1000
    # Pre-define the spectrum configurations for easy lookup
    # Format: "NAME": (Source_Func, Detector_Func, Filter_Name)
    CONFIGURATIONS = {
        "WLED2PD":   ("white_led_spectrum", "photodiode_responsivity", "ALL_PASS"),
        "WLED2PDwF": ("white_led_spectrum", "photodiode_responsivity", "VLC_PASS"),
        "IR2PD":     ("tsff5210_spectrum", "photodiode_responsivity", "ALL_PASS"),
        "IR2PDwF":   ("tsff5210_spectrum", "photodiode_responsivity", "IR_PASS"),
        "WLED2PV":   ("white_led_spectrum", "solar_panel_sensitivity", "ALL_PASS"),
        "SUN2PDwFv": ("sun_spectrum","photodiode_responsivity", "VLC_pass"),
        "SUN2PDwFi": ("sun_spectrum","photodiode_responsivity", "IR_pass"),
        "SUN2PD":    ("sun_spectrum","photodiode_responsivity", "ALL_pass"),
        "SUN2PV":    ("sun_spectrum","solar_panel_sensitivity", "ALL_pass"),

    }

    # --- Helper Methods to reduce Math Duplication ---
    @staticmethod
    def _gaussian(wl: np.ndarray, peak: float, fwhm: float) -> np.ndarray:
        """Generates a Gaussian curve based on Peak and FWHM."""
        sigma = fwhm / (2 * np.sqrt(np.log(2)))
        return np.exp(-(wl - peak)**2.0 / sigma**2.0)

    @staticmethod
    def _poly_response(wl: np.ndarray, coeffs: np.ndarray,
                       l_min_nm: float, l_max_nm: float) -> np.ndarray:
        """Evaluates a polynomial response within a specific nanometer range."""
        wl_nm = wl / 1e-9
        response = np.zeros_like(wl)

        # Boolean mask for valid range
        mask = (wl_nm >= l_min_nm) & (wl_nm <= l_max_nm)

        if np.any(mask):
            # Preserve the specific scaling logic from your original code
            x_scaled = 2 * wl_nm[mask] / (l_min_nm + l_max_nm)
            response[mask] = np.polyval(coeffs, x_scaled)

        return response

    # --- Source Definitions ---
    @classmethod
    def white_led_spectrum(cls, wl: np.ndarray) -> np.ndarray:
        """
        Models a White LED spectrum as a sum of two Gaussians.
        
        Represents the Blue pump (peak ~470nm) and the Yellow Phosphor emission 
        (peak ~600nm).
        """
        
        # Sum of two Gaussians (Blue pump + Phosphor)
        blue = cls._gaussian(wl, peak=470e-9, fwhm=20e-9)
        phosphor = cls._gaussian(wl, peak=600e-9, fwhm=100e-9)
        #plt.plot(wl,blue+phosphor)
        return blue + phosphor

    @classmethod
    def tsff5210_spectrum(cls, wl: np.ndarray) -> np.ndarray:
        """Models the TSFF5210 IR Emitter spectrum (Peak ~870nm)."""
        return cls._gaussian(wl, peak=870e-9, fwhm=40e-9)

    # --- Detector Definitions ---
    @classmethod
    def photodiode_responsivity(cls, wl: np.ndarray) -> np.ndarray:
        """
        Returns the spectral responsivity of the system photodiode.
        
        Based on a 5th-order polynomial fit valid between 330nm and 1090nm.
        """
        coeffs = np.array([-6.39503882, 27.47316339, -45.57791267,
                           36.01964536, -12.8418451, 1.73076976])
        return cls._poly_response(wl, coeffs, 330, 1090)

    @classmethod
    def solar_panel_sensitivity(cls, wl: np.ndarray) -> np.ndarray:
        """
        Returns the spectral sensitivity of the system Solar Panel.
        
        Based on a 6th-order polynomial fit valid between 300nm and 1175nm.
        """
        
        coeffs = np.array([26.78555644, -160.24353775, 381.86564712, -463.07816469,
                           300.12488471, -97.25192023, 12.34949208])
        #plt.plot(wl,cls._poly_response(wl, coeffs, 300, 1175))
        return cls._poly_response(wl, coeffs, 300, 1175)

    @classmethod
    def sun_spectrum(cls,wl:np.ndarray) -> np.ndarray:
        """
        Approximates the Solar Spectrum using Planck's Blackbody radiation law.
        
        Assumes a color temperature T = 5800K. The spectrum is normalized 
        relative to the peak power.
        """
        
        T = 5800
        pmax = 1
        lmax = Constants.bK / T

        def blackbody(lam):
            return (2 * Constants.hP * Constants.c0**2 / lam**5 /
                    (np.exp(Constants.hP * Constants.c0 /
                            (lam * Constants.kB * T)) - 1))

        Pmax = blackbody(lmax)
        xd = pmax * blackbody(wl) / Pmax
        return xd

    @classmethod
    def sun_power(cls):
        """Calculates the total integrated power of the generated solar spectrum."""
        wl = np.linspace(cls.L_MIN, cls.L_MAX, cls.GRID_POINTS)
        xd = cls.sun_spectrum(wl)
        return np.trapz( xd, wl)

    # --- Filters ---
    @staticmethod
    def get_filter_transmission(name: str, wl: np.ndarray) -> np.ndarray:
        """
        Returns the transmission window (0.0 or 1.0) for a given filter name.
        
        Supported filters:
            - 'VLC_PASS': Visible light pass (320nm - 720nm)
            - 'IR_PASS': Infrared pass (770nm - 1100nm)
        """
        
        if name == "VLC_PASS":
            return ((wl >= 320e-9) & (wl <= 720e-9)).astype(float)
        elif name == "IR_PASS":
            return ((wl >= 770e-9) & (wl <= 1100e-9)).astype(float)
        return np.ones_like(wl)

    # --- Core Logic ---
    @classmethod
    def calculate_effective_responsivity(cls,
                                         source_func: Callable,
                                         detector_func: Callable,
                                         filter_name: str = "ALL_PASS") -> float:
        """Calculates overlap integral of Source * Detector * Filter / Integral(Source)."""
        # Create grid once
        wl = np.linspace(cls.L_MIN, cls.L_MAX, cls.GRID_POINTS)

        P_src = source_func(wl)
        R_det = detector_func(wl)
        T_filt = cls.get_filter_transmission(filter_name, wl)

        # Determine denominator (Total Source Power)
        total_power = np.trapz(P_src, wl)

        if total_power == 0:
            return 0.0

        # Determine numerator (detected power)
        detected_power = np.trapz(P_src * R_det * T_filt, wl)

        return float(detected_power / total_power)

    @classmethod
    def get_responsivity_by_name(cls, config_name: str) -> float:
        """Retrieves and calculates responsivity based on the configuration name."""
        if config_name not in cls.CONFIGURATIONS:
            raise ValueError(f"Unknown configuration: {config_name}. Available: {list(cls.CONFIGURATIONS.keys())}")

        src_name, det_name, filt_name = cls.CONFIGURATIONS[config_name]

        # Dynamically fetch the methods from the class
        src_func = getattr(cls, src_name)
        det_func = getattr(cls, det_name)

        return cls.calculate_effective_responsivity(src_func, det_func, filt_name)

In [67]:
import numpy as np

class Surface:
    """
    Represents a rectangular physical surface (e.g., wall, window, ceiling) 
    discretized into a grid of smaller elements.

    This class automatically generates the coordinate points for a grid based on 
    dimensions and resolution, and initializes both Transmitter (Tx) and 
    Receiver (Rx) elements at each grid point to simulate reflection and emission.

    Attributes:
        r_surface (np.ndarray): Coordinates of the center of each grid element (N, 3).
        A (float): Area of a single grid element (patch area).
        Tx_elements (OpticalTxElements): The transmission properties associated with the surface elements.
        Rx_elements (OpticalRxElements): The reception/reflection properties associated with the surface elements.
    """
    def __init__(self, center, dims, const_axis, resolution, nT = np.array([0,0,1]), nR = np.array([0,0,1]),refl = 0.8, type = 'Wall',name=None, P = None):
        """
        Initialize the Surface.

        Args:
            center (array-like): The (x, y, z) center coordinate of the surface.
            dims (tuple): Dimensions (dim_1, dim_2) along the non-constant axes.
            const_axis (int): The axis index (0=x, 1=y, 2=z) that remains constant (normal to surface).
            resolution (tuple): The number of grid points (res_1, res_2) along dim_1 and dim_2.
            nT (array-like, optional): Normal vector for Transmitters (emission direction). Defaults to [0,0,1].
            nR (array-like, optional): Normal vector for Receivers (detection direction). Defaults to [0,0,1].
            refl (float, optional): Reflection coefficient of the surface. Defaults to 0.8.
            type (str, optional): Surface type ('Wall', 'window', etc.). 'window' triggers solar power calc.
            name (str, optional): Label for the surface.
            P (float, optional): Optical power emitted by the surface. If None, defaults to 0 (unless type='window').
        """
        self.r_surface, self.A = self.gen_surface_points(center=center, dims=dims, const_axis = const_axis,
                                                 resolution = resolution)
        self.const_axis = const_axis
        self.name = name

        # Normals will be tiled inside Element init, but we can prep them here
        self.nT = nT
        self.nR = nR
        self.refl = refl
        self.type = type
        self.P = 0 if P is None else P

        if self.type == "window":
            self.P = Constants.pd_peak * self.A * SpectralPhysics.sun_power()


        count = self.r_surface.shape[0]


        self.Tx_elements = OpticalTxElements(r = self.r_surface, n = self.nT, p = self.P, m = 1)
        self.Rx_elements = OpticalRxElements(r = self.r_surface, n = self.nR, A = self.A, refl = self.refl, fov = np.pi/2)



    @staticmethod
    def gen_surface_points(center, dims, const_axis, resolution):
      """
      Generates a mesh grid of points centered at a specific location on a 2D plane 
      in 3D space.

      Args:
          center (array-like): (x, y, z) center.
          dims (tuple): (length_1, length_2) of the rectangle.
          const_axis (int): 0 (YZ plane), 1 (XZ plane), or 2 (XY plane).
          resolution (tuple): (n_points_1, n_points_2) division count.

      Returns:
          tuple: 
              - points (np.ndarray): Array of shape (N, 3) containing grid centers.
              - patch_area (float): The area of a single grid patch.
      """
      dim_1, dim_2 = dims
      res_1, res_2 = resolution

      grid_1 = np.linspace(-dim_1/2 + dim_1/res_1/2, dim_1/2 - dim_1/res_1/2, res_1)
      grid_2 = np.linspace(-dim_2/2 + dim_2/res_2/2, dim_2/2 - dim_2/res_2/2, res_2)
      mesh_1, mesh_2 = np.meshgrid(grid_1, grid_2)
      zeros = np.zeros_like(mesh_1)

      if const_axis == 0: offsets = np.stack([zeros, mesh_1, mesh_2], axis=-1)
      elif const_axis == 1: offsets = np.stack([mesh_1, zeros, mesh_2], axis=-1)
      else: offsets = np.stack([mesh_1, mesh_2, zeros], axis=-1)

      points = center + offsets.reshape(-1, 3)
      patch_area = float((dim_1/res_1)*(dim_2/res_2))
      return points, patch_area

In [68]:
class RoomBuilder:
    """
    Parses a design dictionary to extract and configure physical room parameters.

    This class serves as a parser and state container for the room's geometry,
    surface properties, and special elements (like Windows or RIS units) before
    the actual simulation objects are constructed.

    Attributes:
        design (Dict): The source configuration dictionary.
        L, W, H (float): Room Length, Width, and Height.
        res (int/tuple): Wall grid resolution.
        floor_refl, ceiling_refl, wall_refl (float): Reflectivity coefficients.
        windows (list): List of dictionaries defining window parameters.
        RIS (list): List of dictionaries defining RIS (Reconfigurable Intelligent Surface) parameters.
    """
    def __init__(self, design: Dict, console = False):
        """
        Initialize the RoomBuilder.

        Args:
            design (Dict): Configuration dictionary containing 'environment' keys.
            console (bool): If True, prints extraction details to stdout.
        """
        self.design = design
        self.get_dimensions_and_res(console)
        self.get_reflectivity(console)
        self.get_surfaces_by_type("RIS", console)
        self.get_surfaces_by_type("window", console)


    def get_dimensions_and_res(self, console = False):
        """
        Extracts room dimensions (L, W, H) and meshing resolution.
        
        Args:
            console (bool): Enable debug printing.
        """
        self.L = self.design['environment']['dimensions'][0]
        self.W = self.design['environment']['dimensions'][1]
        self.H = self.design['environment']['dimensions'][2]
        self.res = self.design['environment']['wall_resolution']
        if console:
          print(f"Retrieving design parameters: ")
          print(f"L: {self.L}")
          print(f"W: {self.W}")
          print(f"H: {self.H}")
          print(f"Res: {self.res}")

    def get_reflectivity(self, console = False):
        """
        Extracts reflectivity coefficients for standard room surfaces.
        
        Sets:
            self.floor_refl
            self.ceiling_refl
            self.wall_refl
            self.refl (list of the above)
        """
        self.floor_refl = self.design['environment']['reflectivity']['floor']
        self.ceiling_refl = self.design['environment']['reflectivity']['ceiling']
        self.wall_refl = self.design['environment']['reflectivity']['walls']
        self.refl = [self.floor_refl, self.ceiling_refl, self.wall_refl]

        if isinstance(self.res, int):
          self.res = (self.res, self.res)
        
        if console:
          print(f"Retrieving design parameters: ")
          print(f"Floor Reflectivity: {self.floor_refl}")
          print(f"Ceiling Reflectivity: {self.ceiling_refl}")
          print(f"Wall Reflectivity: {self.wall_refl}")

    def get_surfaces_by_type(self, surface_type, console = False):
      """
      Extracts specific special surface configurations from the design dictionary.

      Populates self.windows or self.RIS depending on the requested type.

      Args:
          surface_type (str): Either 'window' or 'RIS'.
          console (bool): Enable debug printing.

      Raises:
          ValueError: If surface_type is not 'window' or 'RIS'.
      """
      env = self.design.get('environment', {})
      surfaces = env.get('special_surfaces', [])
      if surface_type == 'window':
        self.windows = [s for s in surfaces if s.get('type') == surface_type]
        if console:
          print(f"Found {len(self.windows)} {surface_type} surfaces")
      elif surface_type == 'RIS':
        self.RIS = [s for s in surfaces if s.get('type') == surface_type]
        if console:
          print(f"Found {len(self.RIS)} {surface_type} surfaces")
      else:
        raise ValueError("Invalid surface type. Must be 'window' or 'RIS'.")

In [69]:
class Room:
    """
    Constructs the physical room environment for the simulation.

    This class initializes the standard room surfaces (walls, floor, ceiling) 
    and handles the integration of special surfaces like Windows and RIS 
    (Reconfigurable Intelligent Surfaces) by managing spatial overlaps and 
    reflectivity modifications.

    Attributes:
        windows (list): List of window Surface objects.
        RIS (list): List of RIS Surface objects.
        Tx_wall_elements (OpticalTxElements): Combined transmitter elements for all walls.
        Rx_wall_elements (OpticalRxElements): Combined receiver elements for all walls.
        Tx_RIS_elements (OpticalTxElements): Combined transmitter elements for all RIS units.
        Tx_windows_elements (OpticalTxElements): Combined transmitter elements for all windows.
        h_ww (np.ndarray): Channel gain matrix representing intra-wall reflections (Wall-to-Wall).
    """
    def __init__(self, rb : RoomBuilder, ignore_RIS = False, ignore_windows = False, console = False):
        """
        Initialize the Room using specifications from a RoomBuilder.

        Args:
            rb (RoomBuilder): The builder object containing parsed design parameters.
            ignore_RIS (bool): If True, skips adding RIS units.
            ignore_windows (bool): If True, skips adding windows.
            console (bool): If True, enables debug printing.
        """
        # 1. Initialize Standard Surfaces
        #Helpers
        L = rb.L
        W = rb.W
        H = rb.H
        res = rb.res
        refl = rb.refl

        #CAUTION!!! room.windows & room.RIS is a list of surfaces
        #CAUTION!!! rb.windows & rb.RIS is a list of dictionaries
        self.windows = []
        self.RIS = []

        self.Tx_RIS_elements = None
        self.Tx_windows_elements = None
        self.h_ww = None

        #Create wall surfaces
        self.floor = Surface(np.array([L/2, W/2, 0]), (L, W), 2, res, nR = Constants.zp, nT = Constants.zp, refl=refl[0], name='Floor')
        self.ceiling = Surface(np.array([L/2, W/2, H]), (L, W), 2,  res, nR = Constants.zm, nT = Constants.zm, refl=refl[1], name='Ceiling')
        self.west_wall = Surface(np.array([0, W/2, H/2]), (W, H), 0, res, nR = Constants.xp, nT = Constants.xp, refl=refl[2], name='West Wall')
        self.east_wall = Surface(np.array([L, W/2, H/2]), (W, H), 0, res, nR = Constants.xm, nT = Constants.xm, refl=refl[2], name='East Wall')
        self.south_wall = Surface(np.array([L/2, 0, H/2]), (L, H), 1, res, nR = Constants.yp, nT = Constants.yp, refl=refl[2], name='South Wall')
        self.north_wall = Surface(np.array([L/2, W, H/2]), (L, H), 1, res, nR = Constants.ym, nT = Constants.ym, refl=refl[2], name='East Wall')

        self.walls = [self.floor, self.ceiling, self.west_wall, self.east_wall, self.south_wall, self.north_wall]
        self._build_master_element()

        if not ignore_RIS:
          for ris in rb.RIS:
            ris_surface = Surface(ris['center'], ris['dims'], ris['const_axis'], ris['resolution'], ris['normal'], ris['normal'], ris['reflectivity'], ris['type'], ris['name'])
            self.add_surface(ris_surface)
        if not ignore_windows:
          for window in rb.windows:
            window_surface = Surface(window['center'], window['dims'], window['const_axis'], window['resolution'], window['normal'], window['normal'], window['reflectivity'], window['type'], window['name'])
          self.add_surface(window_surface)
        if console:
          print("Room built successfully!")


    def _build_master_element(self):
      """
      Combines all individual wall surfaces into the main self.wall_elms object.
      
      Merges the elements from Floor, Ceiling, West, East, South, and North walls
      into single unified OpticalTxElements and OpticalRxElements batches.
      """
      Tx_wall_elements = [s.Tx_elements for s in self.walls]
      Rx_wall_elements = [s.Rx_elements for s in self.walls]
      self.Tx_wall_elements = None
      self.Rx_wall_elements = None
      self.Tx_wall_elements = OpticalTxElements.merge(Tx_wall_elements)
      self.Rx_wall_elements = OpticalRxElements.merge(Rx_wall_elements)


    def intra_wall_gains(self):
      """
      Calculates the Line-of-Sight (LoS) channel gains between all wall elements.

      This method computes self.h_ww, which represents the energy transfer 
      matrix from every wall element to every other wall element (used for 
      calculating diffuse reflections).
      """
      tx = self.Tx_wall_elements
      rx = self.Rx_wall_elements
      self.h_ww = Gains.calc_h(tx, rx)

    def plot_surface_addition(self, new_surface, overlap_mask):
        """
        Visualizes the process of adding a new surface (like a Window or RIS) 
        onto an existing wall.

        Plots:
        1.  Blue: Existing wall tiles that remain active.
        2.  Red: Wall tiles being removed/deactivated (because they are behind the new surface).
        3.  Green: The new surface elements.
        4.  Black: A wireframe border around the new surface.

        Args:
            new_surface (Surface): The surface object being added.
            overlap_mask (np.ndarray): Boolean array indicating which wall elements are overlapped.
        """
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')

        # --- 1. Plot Wall Elements (Active vs Removed) ---
        wall_r = self.Rx_wall_elements.r
        kept_r = wall_r[~overlap_mask]
        removed_r = wall_r[overlap_mask]

        ax.scatter(kept_r[:, 0], kept_r[:, 1], kept_r[:, 2],
                   c='blue', alpha=0.05, s=2, label='Existing Wall (Kept)')

        if removed_r.shape[0] > 0:
            ax.scatter(removed_r[:, 0], removed_r[:, 1], removed_r[:, 2],
                       c='red', alpha=0.8, s=10, label='Wall Tiles Removed')

        # --- 2. Plot The New Surface ---
        new_r = new_surface.Tx_elements.r
        ax.scatter(new_r[:, 0], new_r[:, 1], new_r[:, 2],
                   c='green', alpha=0.9, s=15, marker='s', label=f'New {new_surface.name}')

        # --- 3. Draw Borders (New Logic) ---
        # Find which axes vary (not the constant axis)
        const_axis = new_surface.const_axis
        active_axes = [i for i in range(3) if i != const_axis]

        # Calculate limits for the active axes
        limits = {}
        for axis in active_axes:
            vals = new_r[:, axis]
            # Calculate step size (tile width) to find true edge
            unique = np.unique(np.round(vals, 5))
            if len(unique) > 1:
                step = np.mean(np.diff(unique))
            else:
                # If 1x1 grid, infer step from Area (assuming square tile for estimation)
                step = np.sqrt(new_surface.Tx_elements.A[0,0])

            limits[axis] = (vals.min() - step/2, vals.max() + step/2)

        # Construct the 4 corners
        # Start with a base point at the center of the constant axis
        base_val = new_r[0, const_axis]

        # Determine the cycle of corners (rectangular path)
        a1, a2 = active_axes
        min1, max1 = limits[a1]
        min2, max2 = limits[a2]

        # The 5 points to draw the loop (Start -> TopRight -> BottomRight -> BottomLeft -> Start)
        corners_2d = [
            (min1, min2),
            (max1, min2),
            (max1, max2),
            (min1, max2),
            (min1, min2) # Close loop
        ]

        # Convert back to 3D coordinates
        x_line, y_line, z_line = [], [], []
        for (c1, c2) in corners_2d:
            pt = [0, 0, 0]
            pt[const_axis] = base_val
            pt[a1] = c1
            pt[a2] = c2

            x_line.append(pt[0])
            y_line.append(pt[1])
            z_line.append(pt[2])

        ax.plot(x_line, y_line, z_line, color='black', linewidth=3, label='Border')

        # --- Formatting ---
        ax.set_xlabel('X (m)')
        ax.set_ylabel('Y (m)')
        ax.set_zlabel('Z (m)')
        ax.set_title(f"Adding '{new_surface.name}' to Room")
        ax.legend()

        # Fix Aspect Ratio
        max_range = np.array([self.floor.Tx_elements.r[:,0].max(),
                              self.floor.Tx_elements.r[:,1].max(),
                              self.ceiling.Tx_elements.r[:,2].max()]).max()
        ax.set_xlim(0, max_range)
        ax.set_ylim(0, max_range)
        ax.set_zlim(0, max_range)

        plt.show()

    def add_surface(self, new_surface):
        """
        Integrates a special surface (Window or RIS) into the room.

        This method detects which existing wall tiles lie 'behind' the new 
        surface and effectively 'turns them off' by setting their reflectivity 
        to zero. This prevents double-counting reflections at those coordinates.

        Args:
            new_surface (Surface): The surface object to be added.
        """
        if new_surface.type == 'RIS':
            self.RIS.append(new_surface)
            if self.Tx_RIS_elements == None:
                self.Tx_RIS_elements = new_surface.Tx_elements
            else:
                self.Tx_RIS_elements = self.Tx_RIS_elements + new_surface.Tx_elements

        elif new_surface.type == 'window':
            self.windows.append(new_surface)
            if self.Tx_windows_elements == None:
                self.Tx_windows_elements = new_surface.Tx_elements
            else:
                self.Tx_windows_elements = self.Tx_windows_elements + new_surface.Tx_elements

        # Check Overlap
        new_r = new_surface.Tx_elements.r # (N, 3)
        old_r = self.Rx_wall_elements.r   # (M, 3)
        const_axis = new_surface.const_axis
        const_val = new_r[0, const_axis]
        active_axes = [i for i in range(3) if i != const_axis]

        # Mask
        overlap_mask = np.abs(old_r[:, const_axis] - const_val) < 1e-4

        for axis in active_axes:
            vals = new_r[:, axis]
            unique_vals = np.unique(np.round(vals, 5))
            if len(unique_vals) > 1:
                step = np.mean(np.diff(unique_vals))
            else:
                step = np.sqrt(new_surface.Tx_elements.A[0])

            min_edge = np.min(vals) - step/2
            max_edge = np.max(vals) + step/2
            buffer = 1e-5
            is_inside_dim = (old_r[:, axis] > min_edge + buffer) & (old_r[:, axis] < max_edge - buffer)
            overlap_mask = overlap_mask & is_inside_dim

        removed_count = np.sum(overlap_mask)
        print(f"Adding '{new_surface.name}'. Removed {removed_count} tiles.")
        if 0 and removed_count > 0:
            self.plot_surface_addition(new_surface, overlap_mask)

        # Set reflectivity to 0 (note overlap_mask is 1D array of size M)
        # self.Wall_elms.refl is (M, 1)
        self.Rx_wall_elements.refl[overlap_mask,0] = 0

In [70]:
class Gains:
  """
  Calculates optical channel gains (Line-of-Sight, Diffuse, and RIS-assisted) 
  between transmitters and receivers in a specific room environment.
  It can also calculate RF channel gains based on the log-distance path loss
  model that can be computed directly without instantiating the class.

  This class acts as the core physics engine for the channel model, handling 
  vectorized geometric calculations for signal propagation.

  Attributes:
      room (Room): The physical environment containing walls and surfaces.
      masters (OpticalTxElements): The primary light sources (transmitters).
      sensors (OpticalRxElements): The detectors (receivers).
      h_los (np.ndarray): Line-of-Sight channel gain matrix (Nt, Nr).
      h_diff (np.ndarray): Diffuse (multi-bounce) channel gain matrix (Nt, Nr).
      h_ris (np.ndarray): RIS-reflected channel gain matrix (Nt, Nr).
  """
  def __init__(self, room: Room, optRx, optTx):
    self.room = room
    self.masters = optTx
    self.sensors = optRx

    self.h_diff = None
    self.h_los = None



  @staticmethod
  def calc_h_rf(tx, rx, **kwargs):
    """
    Calculates the path loss for RF signals based on the log-distance path loss model.
    
    Formula: PL(d) = 10*n*log10(d) + PL_ref + 10*k*log10(f) + X_sigma
    
    Args:
        tx (RFTxElements): RF Transmitter elements.
        rx (RFRxElements): RF Receiver elements.
        **kwargs: Optional channel model coefficients.
            - n (float): Path loss exponent (default: 1.46)
            - pl_ref (float): Reference path loss constant (default: 34.62)
            - k (float): Frequency dependence coefficient (default: 2.03)
            - f (float): Frequency in GHz (default: 2.45)
            - sigma (float): Shadowing or additional loss term (default: 3.76)
            - sigma_factor (float): Multiplier for sigma (default: 2)

    Returns:
        np.ndarray: Path loss values in dB.
    """
    # Extract coefficients from kwargs with existing values as defaults
    n = kwargs.get('n', 1.46)
    pl_ref = kwargs.get('pl_ref', 34.62)
    k = kwargs.get('k', 2.03)
    f = kwargs.get('f', 2.45)
    sigma = kwargs.get('sigma', 3.76)
    sigma_factor = kwargs.get('sigma_factor', 2)

    # Calculate distance
    D_tx_rx = -(tx.r[:, None, :] - rx.r[None, :, :])
    d = np.linalg.norm(D_tx_rx, axis=2)

    # Calculate Path Loss
    return (10 * n * np.log10(d)) + pl_ref + (10 * k * np.log10(f)) + (sigma_factor * sigma)

  @staticmethod
  def calc_h(tx, rx):
    """
    Calculates channel gain matrix H (Nt, Nr) for optical links.

    Implements the standard Lambertian emission model.
    Gain = (A * (m+1) * cos(phi)^m * cos(psi)) / (2 * pi * d^2)

    Where:
        phi: Irradiance angle (at Tx)
        psi: Incidence angle (at Rx)
        m: Lambertian order
        d: Distance
        A: Detector Area

    Assumes tx and rx properties are already formatted as (N,3) or (N,1).

    Args:
        tx (Elements): Transmitter batch (Sources or Wall patches).
        rx (Elements): Receiver batch (Detectors or Wall patches).

    Returns:
        np.ndarray: Gain matrix of shape (Nt, Nr).
    """

    # Extract positions: (Nt, 3) and (Nr, 3)
    # No reshapes needed, Element class guarantees (N, 3)

    # Distance Vector (Nt, Nr, 3)
    # Broadcasting: (Nt, 1, 3) - (1, Nr, 3)
    D_tx_rx = -(tx.r[:, None, :] - rx.r[None, :, :])

    # Distance Norm (Nt, Nr)
    D_tx_rx_norm = np.linalg.norm(D_tx_rx, axis=2)

    # Unit Vectors (Nt, Nr, 3)
    D_tx_rx_unit = D_tx_rx / D_tx_rx_norm[..., None]

    # --- Cosines ---
    # tx.nT is (Nt, 3). Reshape to (Nt, 1, 3) for broadcasting
    # rx.nR is (Nr, 3). Reshape to (1, Nr, 3) for broadcasting

    # Cosine of irradiance angle (Tx -> Rx)
    cos_irr = np.maximum(0, np.sum(D_tx_rx_unit * tx.n[:, None, :], axis=2)) # (Nt, Nr)

    # Cosine of incidence angle (Rx <- Tx)
    # Note negative sign on D vector for Rx perspective
    cos_inc = np.maximum(0, np.sum(-D_tx_rx_unit * rx.n[None, :, :], axis=2)) # (Nt, Nr)

    # --- FOV Logic ---
    inc_angle = np.arccos(cos_inc) # (Nt, Nr)

    # rx.fov is (Nr, 1). Transpose to (1, Nr) to match columns
    fov_mask_pd = (inc_angle <= rx.fov.T) # (Nt, Nr)

    # rx.type_Rx is (Nr, 1). Transpose to (1, Nr) for boolean usage
    is_sp = (rx.type_Rx.T == 1) # (1, Nr) boolean mask

    # --- Gain Calculation ---
    # rx.A is (Nr, 1) -> Transpose to (1, Nr)
    # tx.m is (Nt, 1) -> Matches rows (Nt) perfectly

    h = (
        rx.A.T * (tx.m + 1) * (cos_irr ** tx.m) * cos_inc
        / (2 * np.pi * D_tx_rx_norm**2)
    ) # Result is (Nt, Nr)

    # Apply FOV Mask (Only for standard PDs, i.e., NOT solar panels)
    # is_sp is (1, Nr). ~is_sp masks columns.
    # We broadcast mask (Nt, Nr) against columns where ~is_sp is True.
    h = np.where(~is_sp, h * fov_mask_pd, h)

    # Apply Solar Panel Efficiency if any exist
    if np.any(is_sp):
        sp_factor = solar_panel_angular_efficiency(cos_inc)
        h = np.where(is_sp, h * sp_factor, h)

    h = np.nan_to_num(h, nan=0)
    return h


  def los_channel_gains(self):
    """Computes the Line-of-Sight (LoS) gain matrix between Masters and Sensors."""
    self.h_los = self.calc_h(tx=self.masters, rx=self.sensors)

  def diffuse_channel_gains(self, bounces = 4):
    """
    Computes the Non-Line-of-Sight (Diffuse) gain matrix using an iterative 
    multi-bounce approach.

    Algorithm:
    1. Calculate Source -> Wall gains (h_mw).
    2. Calculate Wall -> Sensor gains (h_ws).
    3. Calculate Wall -> Wall gains (h_ww) if not already done.
    4. Iteratively propagate power between wall elements for 'k' bounces.
    
    Args:
        bounces (int): Number of reflection bounces to simulate.
    """
    self.h_mw = self.calc_h(tx = self.masters, rx = self.room.Rx_wall_elements)
    self.h_ws = self.calc_h(tx = self.room.Tx_wall_elements, rx= self.sensors)

    if self.room.h_ww is None:
        self.room.intra_wall_gains()

    R = np.diag(self.room.Rx_wall_elements.refl.flatten())
    current_wall_power = self.h_mw @ R
    m = self.masters.r.shape[0]
    s = self.sensors.r.shape[0]
    H_diffuse_total = np.zeros((m, s))

    for k in range(1, bounces + 1):
        bounce_contribution = current_wall_power @ self.h_ws
        H_diffuse_total += bounce_contribution
        if k < bounces:
            current_wall_power = (current_wall_power @ self.room.h_ww) @ R

    self.h_diff = H_diffuse_total

  def ris_channel_gains(self):
      """
      Computes the channel gain via a Reconfigurable Intelligent Surface (RIS).
      
      Calculates the two-hop path: Source -> RIS Element -> Sensor.
      This ignores phase shifts (pure amplitude gain), assuming perfect phase alignment 
      or amplitude-only modeling depending on context.
      """
      self.h_ris = np.zeros((self.masters.N, self.sensors.N))
      if self.room.Tx_RIS_elements is None:
          print("You need to add a RIS surface first")
          return
      else:
          self.no_masters = self.masters.r.shape[0]
          self.no_sensors = self.sensors.r.shape[0]
          self.no_ris_elems = self.room.Tx_RIS_elements.r.shape[0]
          self.D_tx_ris = np.zeros([self.no_masters,self.no_ris_elems,3])
          self.D_tx_ris_unit = np.zeros([self.no_masters,self.no_ris_elems,3])
          self.D_tx_ris_norm = np.zeros([self.no_masters,self.no_ris_elems])
          self.D_rx_ris = np.zeros([self.no_sensors,self.no_ris_elems,3])
          self.D_rx_ris_unit = np.zeros([self.no_sensors,self.no_ris_elems,3])
          self.D_rx_ris_norm = np.zeros([self.no_sensors,self.no_ris_elems])
          self.d = np.zeros([self.no_masters,self.no_sensors,self.no_ris_elems])
          self.hris = np.zeros([self.no_masters,self.no_sensors,self.no_ris_elems])

          r_master = self.masters.r
          r_sensor = self.sensors.r
          r_ris = self.room.Tx_RIS_elements.r

          #BROADCASTING RIS CALCULATIONS
          self.D_tx_ris = -1*(r_master[:,None,:] - r_ris[None,:,:])
          self.D_tx_ris_norm = np.linalg.norm(self.D_tx_ris,axis=2)
          self.D_tx_ris_unit = self.D_tx_ris/self.D_tx_ris_norm[...,None]
          self.D_rx_ris = -1*(r_sensor[:,None,:] - r_ris[None,:,:])
          self.D_rx_ris_norm = np.linalg.norm(self.D_rx_ris,axis=2)
          self.D_rx_ris_unit = self.D_rx_ris/self.D_rx_ris_norm[...,None]
          self.cos_irr = np.maximum(0,np.sum(self.D_tx_ris_unit * self.masters.n.reshape(-1, 1, 3), axis=2))
          self.cos_inc = np.maximum(0,np.sum(self.D_rx_ris_unit * self.sensors.n.reshape(-1, 1, 3), axis=2))
          self.d = self.D_tx_ris_norm[:,None,:] + self.D_rx_ris_norm[None,...]
          self.hris = self.sensors.A[None,...] * (self.masters.m[...,None]+1) * self.cos_irr[:,None,:]**self.masters.m[...,None] * self.cos_inc[None,...]/(2*np.pi * self.d)
          inc_angle = np.arccos(self.cos_inc) # (Nt, Nr)
          is_sp = (self.sensors.type_Rx == 1)
          fov_mask_pd = (inc_angle <= self.sensors.fov)
          self.hris = np.where(~is_sp, self.hris * fov_mask_pd, self.hris)
          if np.any(is_sp):
              sp_factor = solar_panel_angular_efficiency(self.cos_inc)
              h = np.where(is_sp, self.hris * sp_factor, self.hris)
          self.h_ris = np.sum(self.hris,axis=2)

In [71]:
class NodeBuilder():
  """
  Parses design configurations to prepare parameters for node creation.

  This class acts as a factory helper, extracting raw data from the design dictionary
  and normalizing it (e.g., reshaping arrays, applying defaults) so that 
  Master, Sensor, or AmbientNode objects can be instantiated cleanly.
  """
  def __init__(self, design: Dict, node_type,console = False):
    """
    Initialize the NodeBuilder.

    Args:
        design (Dict): The main configuration dictionary containing a "nodes" key.
        node_type (str): The specific category to build ('masters', 'sensors', or 'ambient_nodes').
        console (bool): If True, enables debug printing (currently unused).
    """
    self.design = design
    self.node_type = node_type
    self.get_node_params(node_type, console)
    self.sanity_check()

  def get_node_params(self, node_type, console = False):
    """
    Extracts params for specific node types from the design dictionary.

    Populates instance attributes including physical location, orientation (nT/nR),
    electrical properties (battery, power), and optical properties (FOV, bandwidth).

    Args:
        node_type (str): 'masters', 'sensors', or 'ambient_nodes'.
    """
    nodes = self.design["nodes"][node_type].get("Type") #returns None if it is not there
    self.node_type = node_type
    self.rx_area = self.design["nodes"][node_type].get("rx_area")
    self.nT = self.design["nodes"][node_type].get("nT")
    self.nR = self.design["nodes"][node_type].get("nR")
    self.m = self.design["nodes"][node_type].get("m", 1)
    self.tx_power = self.design["nodes"][node_type].get("tx_power")
    self.rx_type = self.design["nodes"][node_type].get("rx_type")
    self.uplink_type = self.design["nodes"][node_type].get("uplink_type",0)
    self.battery_capacity_J = self.design["nodes"][node_type].get("battery_capacity_J")
    self.initial_charge = self.design["nodes"][node_type].get("initial_charge")
    self.positions = self.design["nodes"][node_type].get("positions").reshape(-1,3)
    self.FOV = self.design["nodes"][node_type].get("FOV", np.pi/2)
    #self.BW = self.design["nodes"][node_type].get("BW",10e3)
    self.tia = self.design.get("TIA",None)
    self.IR_tx_power = self.design["nodes"][node_type].get("IR_tx_power",0)
    self.RF_tx_power = self.design["nodes"][node_type].get("RF_tx_power",0)


    energy_prof = self.design.get('energy_profile', {})
    comm_cfg = energy_prof.get('communication', {})

    self.n_sp_d = comm_cfg.get("n_sp_d", 0.4)
    self.n_sp_u = comm_cfg.get("n_sp_d", 0.4)
    self.Rb_down = comm_cfg.get("Rb_down", 20e3)
    self.Rb_up = comm_cfg.get('Rb_up', 100e3)  

    
    if node_type == "sensors":
      self.VLC_pass_filter = self.design["nodes"][node_type].get("VLC_pass_filter", True)
          
    elif node_type == "masters":
      self.IR_pass_filter = self.design["nodes"][node_type].get("IR_pass_filter", True)
      self.sensitivity = self.design["nodes"][node_type].get("sensitivity", -100)
    else:
      self.IR_pass_filter = None
      self.VLC_pass_filter = None
  def sanity_check(self):
    """
    Validates parameter dimensions and filters optical properties for hybrid networks.

    Logic:
        - If node_type is 'sensors', it checks 'uplink_type' (0=Optical, 1=RF).
        - It ensures that optical transmitter properties (nT, m) are only assigned 
          to sensors that actually have optical uplinks.
        - If 'nT' or 'm' arrays match the total number of positions but include RF nodes,
          this method slices them to keep only the optical nodes' parameters to avoid shape mismatches.
    """
    #size sanity check for different input styles for uplinks
    if self.node_type != "sensors":
      pass
    else:
      self.uplink_type = to_scal_Nx1(self.positions.reshape(-1,3).shape[0], self.uplink_type).flatten()
      self.no_optical_uplinks = np.where(np.array([self.uplink_type])==0)[0].size
      self.no_RF_uplinks = np.where(np.array([self.uplink_type])==1)[0].size
      self.nT = np.array(self.nR)
      self.m = np.array(self.m)


      if self.nT.reshape(-1,3).shape[0] != 1 and self.nT.reshape(-1,3).shape[0] != self.no_optical_uplinks:
        if self.nT.reshape(-1,3).shape[0] == self.positions.reshape(-1,3).shape[0]:
          nT_x = self.nT[self.uplink_type == 0]
          self.nT = nT_x


      if self.m.reshape(-1,1).shape[0] != 1 and self.m.reshape(-1,1).shape[0] != self.no_optical_uplinks:
        if self.nT.reshape(-1,1).shape[0] == self.positions.reshape(-1,3).shape[0]:
          m_x = self.m[self.uplink_type == 0]
          self.m = m_x

In [72]:
class SNManager:
    """
    Sensor Node Manager.
    
    Orchestrates the initialization of sensor nodes and computes the electro-optical 
    conversion efficiency based on spectral matching.
    
    This class handles the creation of physical simulation elements (Optical/RF 
    transceivers) and pre-calculates the **Effective Responsivity** ($R_{eff}$) 
    for both Signal (Downlink) and Noise (Solar Background) sources.

    Attributes:
        tia (TIA): Transimpedance Amplifier model defining electrical noise and gain.
        no_sensors (int): Total count of receiver nodes.
        ORx_elements (OpticalRxElements): Photodiode and Solar Panel receiver batch.
        OTx_elements (OpticalTxElements): Optical uplinks (IR emitters).
        RFTx_elements (RFTxElements): RF uplinks (if hybrid system is active).
        c_d (np.ndarray): Signal Effective Responsivity vector [A/W].
        c_d_n (np.ndarray): Noise/Background Spectral Matching Factor.
    """
    def __init__(self,nb: NodeBuilder):
      """
      Initialize the Sensor Manager.

      Segregates nodes into Optical and RF categories and initializes the 
      spectral response matrices.

      Args:
          nb (NodeBuilder): The configured builder containing raw node parameters.
      """
      self.tia = TIA(**nb.tia)
      self.no_sensors = nb.positions.shape[0]
      self.Rb_up = nb.Rb_up   #transform shape in PhyNet
      self.n_sp_u = nb.n_sp_u #transform shape in Phynet
      self.rf_flag = 0
      self.ir_flag = 0
      self.ORx_elements = OpticalRxElements( r = nb.positions, n = nb.nR, type_Rx = nb.rx_type, fov = nb.FOV, A = nb.rx_area)
      if nb.no_optical_uplinks > 0:
        self.OTx_elements = OpticalTxElements( r = nb.positions[nb.uplink_type == 0 ], n = nb.nT, m = nb.m, p = nb.IR_tx_power)
        self.ir_flag = nb.no_optical_uplinks
      if nb.no_RF_uplinks > 0:
        self.RFTx_elements = RFTxElements(r = nb.positions[nb.uplink_type == 1], p = to_scal_Nx1(nb.no_RF_uplinks,nb.RF_tx_power))
        self.rf_flag = nb.no_RF_uplinks
      #make the masks for the effective responsivity calculations
      self.mask_VLC_filter = normalize_bool_array(nb.VLC_pass_filter,self.no_sensors) & (nb.rx_type.reshape(-1,) == 0)
      self.mask_pd_no_filter = ~normalize_bool_array(nb.VLC_pass_filter,self.no_sensors) & (nb.rx_type.reshape(-1,) == 0)
      self.mask_pv = (nb.rx_type.reshape(-1,) == 1)

      self.c_d = np.zeros([self.no_sensors]) #Array to hold effective responsivity values
      self.c_d_n = np.zeros([self.no_sensors]) #Array to hold effective responsivity values

      self.downlink_effective_responsivity()


    def downlink_effective_responsivity(self):
      """
      Computes the Effective Responsivity ($R_{eff}$) via the spectral overlap integral.

      The received optical power is converted to current based on the matching 
      between the Source Spectrum ($S(\lambda)$), Filter Transmission ($T(\lambda)$), 
      and Detector Responsivity ($R(\lambda)$).

      **1. Signal Responsivity ($c_d$):**
      $$c_d = \\frac{\\int S_{LED}(\\lambda) \\cdot T_{filt}(\\lambda) \\cdot R_{det}(\\lambda) d\\lambda}{\\int S_{LED}(\\lambda) d\\lambda}$$
      
      **2. Noise Responsivity ($c_{d,n}$):**
      Calculated similarly using the Solar Spectrum ($S_{Sun}(\\lambda)$) as the source. 
      This determines the DC background current ($I_{bg}$) which generates Shot Noise.

      **Special Case - Solar Panels (PV):**
      For PVs, the responsivity is normalized against the solar spectrum to model 
      performance relative to standard STC conditions:
      $$c_{d, PV} = \\frac{R_{eff}(LED \\to PV)}{R_{eff}(Sun \\to PV)}$$
      """
      self.c_d[self.mask_VLC_filter] = SpectralPhysics.get_responsivity_by_name("WLED2PDwF")
      self.c_d[self.mask_pv] = SpectralPhysics.get_responsivity_by_name("WLED2PV")/SpectralPhysics.get_responsivity_by_name("SUN2PV") #different approach for PVs
      self.c_d[self.mask_pd_no_filter] = SpectralPhysics.get_responsivity_by_name("WLED2PD")
      #for noise calculations - sun to pd / pv
      self.c_d_n[self.mask_VLC_filter] = SpectralPhysics.get_responsivity_by_name("SUN2PDwFv")
      self.c_d_n[self.mask_pd_no_filter] = SpectralPhysics.get_responsivity_by_name("SUN2PD")
      self.c_d_n[self.mask_pv] = 1 # for irradiance the sun's spectrum is already taken into account

In [73]:
class MNManager:
    """
    Master Node Manager (Base Stations / Access Points).

    Manages the physical and spectral properties of the central transmitters (Masters).
    Unlike sensors, Masters typically act as the primary sources of Downlink optical power 
    (Visible Light) and receivers of Uplink data (Infrared).

    Attributes:
        tia (TIA): Transimpedance Amplifier model for the Master's receiver circuit.
        no_masters (int): Total number of master nodes.
        sensitivity (np.ndarray): Optical sensitivity.
        ORx_elements (OpticalRxElements): The Master's receiver hardware (detecting Uplink).
        OTx_elements (OpticalTxElements): The Master's transmitter hardware (sending Downlink).
        c_d (np.ndarray): Signal Effective Responsivity vector ($A/W$) for Uplink.
        c_d_n (np.ndarray): Noise Effective Responsivity vector ($A/W$).
    """
    def __init__(self,nb: NodeBuilder):
      """
      Initialize the Master Node Manager.

      Configures the Masters with specific Bandwidth, Sensitivity, and Optical Filters 
      (typically IR-pass filters to reject visible downlink reflections).

      Args:
          nb (NodeBuilder): The configured builder containing raw node parameters.
      """
      self.tia = TIA(**nb.tia)
      self.no_masters = nb.positions.shape[0]
      
      self.Rb_down = nb.Rb_down  #transform shape in Phynet
      self.n_sp_d = nb.n_sp_d  #transform shape in Phynet
       
      
      self.sensitivity = to_scal_Nx1(self.no_masters,nb.sensitivity)
      #make the masks for the effective responsivity calculations
      self.mask_IR_filter = normalize_bool_array(nb.IR_pass_filter,self.no_masters)
      self.mask_pd_no_filter = ~normalize_bool_array(nb.IR_pass_filter,self.no_masters)

      self.ORx_elements = OpticalRxElements( r = nb.positions, n = nb.nR, type_Rx = 0, fov = nb.FOV, A = nb.rx_area)
      self.OTx_elements = OpticalTxElements( r = nb.positions, n = nb.nT, m = nb.m, p = nb.tx_power)


      self.c_d = np.zeros([self.no_masters]) #Array to hold effective responsivity values
      self.c_d_n = np.zeros([self.no_masters]) #Array to hold effective responsivity values

      self.uplink_effective_responsivity()


    def uplink_effective_responsivity(self):
      """
      Computes the Effective Responsivity ($c_d$) for the Uplink Channel.

      This defines how efficiently the Master's Photodiode converts the received 
      Infrared (IR) signal from sensors into electrical current.

      **Spectral Overlap Integral:**
      $$c_d = \\frac{\\int S_{IR-Tx}(\\lambda) \\cdot T_{IR-Filter}(\\lambda) \\cdot R_{PD}(\\lambda) d\\lambda}{\\int S_{IR-Tx}(\\lambda) d\\lambda}$$

      Where:
      * $S_{IR-Tx}$: Spectrum of the Sensor's IR LED (e.g., TSFF5210).
      * $T_{IR-Filter}$: Transmission curve of the Master's optical filter.
      * $R_{PD}$: Spectral responsivity of the Master's Photodiode.
      """
      self.c_d[self.mask_IR_filter] = SpectralPhysics.get_responsivity_by_name("IR2PDwF")
      self.c_d[self.mask_pd_no_filter] = SpectralPhysics.get_responsivity_by_name("IR2PD")

      self.c_d_n[self.mask_IR_filter] = SpectralPhysics.get_responsivity_by_name("IR2PDwF")
      self.c_d_n[self.mask_pd_no_filter] = SpectralPhysics.get_responsivity_by_name("IR2PD")

In [74]:
class ANManager:
    """
    Ambient Node Manager.

    Manages auxiliary light sources in the environment that are not data-carrying 
    Masters but contribute to the total optical power (and potentially interference). 
    These could represent standard room lighting (ceiling lamps, desk lamps) that 
    act as optical interference sources for the sensors.

    Attributes:
        OTx_elements (OpticalTxElements): The transmitter elements representing the ambient light sources.
    """
    def __init__(self,nb: NodeBuilder):
      """
      Initialize the Ambient Node Manager.

      Creates the optical transmitter elements for ambient sources based on the 
      builder configuration.

      Args:
          nb (NodeBuilder): The configured builder containing ambient node parameters.
      """
      self.OTx_elements = OpticalTxElements( r = nb.positions, n = nb.nT, m = nb.m, p = nb.tx_power)

In [75]:
class oPhyGains:
  """
  Optical Physical Layer Gain Calculator.

  This class acts as the central physics engine for the link budget. It orchestrates the 
  calculation of geometric channel states and converts them into received optical power 
  and electrical photocurrents.

  It handles three distinct propagation environments:
  1.  **Downlink (Visible Light):** From Master LEDs to Sensor Photodiodes/PVs.
  2.  **Uplink (Infrared/RF):** From Sensor IR LEDs (or RF antennas) to Master receivers.
  3.  **Ambient (Interference):** From Artificial sources (Lamps) and Natural sources (Windows/Sun).

  Attributes:
      h_*: Geometric Channel Gains (dimensionless or path loss).
      p_*: Received Optical Power [Watts].
      i_*: Induced Photocurrent [Amps].
  """
  def __init__(self, room, masters: MNManager, sensors: SNManager, ambient: ANManager):
    """
    Initialize the Physics Engine.

    Args:
        room (Room): The physical environment (walls, windows, RIS).
        masters (MNManager): Master nodes (Base Stations).
        sensors (SNManager): Sensor nodes (User Terminals).
        ambient (ANManager): Ambient light sources (Interference).
    """
    self.room = room
    self.mn = masters
    self.sn = sensors
    self.ambient = ambient
    self.compute_gains()
    self.compute_ambient()


  def compute_gains(self):
    """
    Computes the Geometric Channel Gains ($H$) for all active links.

    This method isolates the geometry-dependent components of the link.
    
    **Downlink Channels:**
    * $H_{LOS}$: Direct Line-of-Sight gain (Lambertian).
    * $H_{Diff}$: Non-Line-of-Sight gain via wall reflections (Multi-bounce).
    * $H_{RIS}$: Controlled reflection gain via Intelligent Surfaces.

    **Uplink Channels:**
    * Checks `ir_flag` to compute Optical Uplink gains (IR).
    * Checks `rf_flag` to compute RF Path Loss (if hybrid system).
    """
    #downlink gains

    gains = Gains(self.room, self.sn.ORx_elements, self.mn.OTx_elements)
    gains.los_channel_gains()
    gains.diffuse_channel_gains()
    gains.ris_channel_gains()

    self.h_d_los = gains.h_los
    self.h_d_diff = gains.h_diff
    self.h_d_ris = gains.h_ris

    #uplink gains
    if self.sn.ir_flag > 0:
      gains = Gains(self.room, self.mn.ORx_elements, self.sn.OTx_elements)
      gains.los_channel_gains()
      gains.diffuse_channel_gains()
      gains.ris_channel_gains()

      self.h_u_los = gains.h_los
      self.h_u_diff = gains.h_diff
      self.h_u_ris = gains.h_ris

    if self.sn.rf_flag > 0:
      self.h_u_rf = Gains.calc_h_rf(self.sn.RFTx_elements, self.mn.ORx_elements)


  def compute_downlink(self):
    """
    Calculates the Downlink Optical Power and Photocurrent.

    **1. Received Power ($P_{rx}$):**
    Scales the geometric gain $H$ by the transmitted optical power ($P_{tx}$).
    $$P_{rx} = H \cdot P_{tx}$$

    **2. Signal Current ($I_{sig}$):**
    Converts optical power to current using the Effective Responsivity ($c_d$).
    $$I_{sig} = P_{rx} \cdot c_d$$
    
    The total signal current is the sum of LoS, Diffuse, and RIS contributions.
    """

    self.p_d_los = self.h_d_los * self.mn.OTx_elements.p
    self.p_d_diff = self.h_d_diff * self.mn.OTx_elements.p
    self.p_d_ris = self.h_d_ris * self.mn.OTx_elements.p

    self.i_d_los = self.p_d_los * self.sn.c_d
    self.i_d_diff = self.p_d_diff * self.sn.c_d
    self.i_d_ris = self.p_d_ris * self.sn.c_d

    self.i_d_signal = self.i_d_los + self.i_d_diff + self.i_d_ris

  def compute_uplink(self):
    """
    Calculates the Uplink Optical Power and Photocurrent.

    Similar to downlink, but flows from Sensors to Masters.
    Uses `sn.OTx_elements.p` (Sensor Tx Power) and `mn.c_d` (Master Responsivity).
    """

    self.p_u_los = self.h_u_los * self.sn.OTx_elements.p
    self.p_u_diff = self.h_u_diff * self.sn.OTx_elements.p
    self.p_u_ris = self.h_u_ris * self.sn.OTx_elements.p

    self.i_u_los = self.p_u_los * self.mn.c_d
    self.i_u_diff = self.p_u_diff * self.mn.c_d
    self.i_u_ris = self.p_u_ris * self.mn.c_d

    self.i_u_signal = self.i_u_los + self.i_u_diff + self.i_u_ris

  def compute_ambient(self):
    """
    Computes Optical Interference (Shot Noise precursors).

    Calculates the background current generated by non-signal sources.

    **1. Artificial Ambient (Lamps):**
    * Models interference from room lighting ($ix$).
    * Calculated for both Downlink (at Sensors) and Uplink (at Masters).
    * Uses Signal Responsivity ($c_d$) assuming lamps have similar spectra to white LEDs.

    **2. Natural Ambient (Windows/Sun):**
    * Models strong background interference from sunlight ($is$).
    * Uses Noise Responsivity ($c_{d,n}$) because solar spectrum differs from LED spectrum.
    * Sums contributions from all window tiles to get total background current.
    """

    self.ix_d_noise = np.zeros((1, self.sn.ORx_elements.N))
    self.is_d_noise = np.zeros((1, self.sn.ORx_elements.N))

    # Uplink noise (at Masters)
    self.ix_u_noise = np.zeros((1, self.mn.ORx_elements.N))
    self.is_u_noise = np.zeros((1, self.mn.ORx_elements.N))  
      
    if self.ambient.OTx_elements is not None:
      gains_down = Gains(self.room, self.sn.ORx_elements, self.ambient.OTx_elements)

      gains_down.los_channel_gains()
      gains_down.diffuse_channel_gains()

      self.hx_d_los = gains_down.h_los
      self.hx_d_diff = gains_down.h_diff

      self.px_d_los =  self.hx_d_los * self.ambient.OTx_elements.p
      self.px_d_diff = self.hx_d_diff * self.ambient.OTx_elements.p

      self.ix_d_los = self.px_d_los * self.sn.c_d
      self.ix_d_diff = self.px_d_diff * self.sn.c_d
      self.ix_d_noise = (self.ix_d_los + self.ix_d_diff).reshape(-1,self.sn.no_sensors)

      gains_up = Gains(self.room, self.mn.ORx_elements, self.ambient.OTx_elements)

      gains_up.los_channel_gains()
      gains_up.diffuse_channel_gains()

      self.hx_u_los = gains_up.h_los
      self.hx_u_diff = gains_up.h_diff

      self.px_u_los =  self.hx_u_los * self.ambient.OTx_elements.p
      self.px_u_diff = self.hx_u_diff * self.ambient.OTx_elements.p

      self.ix_u_los = self.px_u_los * self.mn.c_d
      self.ix_u_diff = self.px_u_diff * self.mn.c_d
      self.ix_u_noise = (self.ix_u_los + self.ix_u_diff).reshape(-1,self.mn.no_masters)


    if self.room.Tx_windows_elements is not None:
      #windows elems should be initialized correctly
      gains_s_up = Gains(self.room, self.mn.ORx_elements,self.room.Tx_windows_elements)

      gains_s_up.los_channel_gains()
      gains_s_up.diffuse_channel_gains()

      self.hs_u_los = gains_s_up.h_los
      self.hs_u_diff = gains_s_up.h_diff

      self.ps_u_los = self.hs_u_los * self.room.Tx_windows_elements.p
      self.ps_u_diff = self.hs_u_diff * self.room.Tx_windows_elements.p

      self.is_u_los = self.ps_u_los * self.mn.c_d_n
      self.is_u_diff = self.ps_u_diff * self.mn.c_d_n

      self.is_u_los_sum = np.sum(self.is_u_los,axis = 0)
      self.is_u_diff_sum = np.sum(self.is_u_diff, axis = 0)

      self.is_u_noise = (self.is_u_los_sum + self.is_u_diff_sum).reshape(-1,self.mn.no_masters)


      gains_s_down = Gains(self.room, self.sn.ORx_elements,self.room.Tx_windows_elements)

      gains_s_down.los_channel_gains()
      gains_s_down.diffuse_channel_gains()

      self.hs_d_los = gains_s_down.h_los
      self.hs_d_diff = gains_s_down.h_diff

      self.ps_d_los = self.hs_d_los * self.room.Tx_windows_elements.p
      self.ps_d_diff = self.hs_d_diff * self.room.Tx_windows_elements.p

      self.is_d_los = self.ps_d_los * self.sn.c_d_n
      self.is_d_diff = self.ps_d_diff * self.sn.c_d_n

      self.is_d_los_sum = np.sum(self.is_d_los,axis = 0)
      self.is_d_diff_sum = np.sum(self.is_d_diff, axis = 0)

      self.is_d_noise = (self.is_d_los_sum + self.is_d_diff_sum).reshape(-1,self.sn.no_sensors)

In [76]:
class PV:
    """
    Photovoltaic Cell Model.
    
    This class implements a comprehensive physical and electrical model of a solar cell,
    spanning DC operating point analysis, AC small-signal characteristics, and 
    frequency-dependent noise modeling.
    """
    def __init__(self, Gsignal, Gamb, unscaled=True, run = True, **kwargs):
        """
        Args:
            Gsignal: Signal Irradiance (W/m^2)
            Gamb: Ambient Irradiance (W/m^2)
            unscaled: If True, scales Rs, Rsh, Jsc by Area.
            **kwargs: Overrides for circuit (Rs, Rsh...) or physics (Na, Nd...) params.
        """
        # 1. Setup Input Vectors
        self.Gamb = np.array(Gamb).reshape(-1, 1)
        self.Gsignal = np.array(Gsignal).reshape(-1, 1)
        self.no_pv = self.Gamb.shape[0]
        self.Gref = 1000.0

        # 2. Initialize Parameters (Circuit + Physics)
        # 
        self._init_params(kwargs)

        # 3. Scale Circuit Parameters
        if unscaled:
            self._scale_params()

        # 4. DC Operating Point & Bias
        self._calc_dc_bias()

        # 5. IV Curve Calculation (Analytic Lambert W το avoid loops)
        # Sweep 0 to Voc (Open Circuit)
        # 
        self.V = self.Voc * np.linspace(0, 1, 100).reshape(1, -1) # Shape: (1, 100) -> broadcasts to (N, 100)
        self.I = self.pv_current(self.V)
        self.I[self.I <= 0] = 1e-20  # To avoid negative current

        # 6. DC Performance Metrics
        self.P = self.I * self.V
        self.ind = np.argmax(self.P, axis=1) # Index of MPP
        self.Pmax = np.take_along_axis(self.P, self.ind[:, None], axis=1)
        self.Rl = self.V / self.I

        # 7. Small Signal Parameters (Dynamic Resistance)
        # ID is diode current
        self.ID = self.I0 * np.exp((self.V + self.I * self.Rs) / (self.n * self.Vt))
        self.r = (self.n * self.Vt) / self.ID
        self.iac = self.Iph * self.Gsignal/self.Gamb

        # 8. Frequency & Noise Setup
        self.calc_capacitance() # Uses self.Na, self.Nd from kwargs
        self.find_bw()

        # Generate Frequency Vector based on Bandwidth
        # Creates frequency sweep from 100Hz up to cut-off freq of each panel
        self.bw_ind = np.take_along_axis(self.BW, self.ind[:, None], axis=1) # (N, 1)
        f_steps = 600
        self.f = np.linspace(100, self.bw_ind.flatten(), f_steps).T

        #If default frequency vector is used we can run everything here.
        #For other freq intervals use run = False at object initiation and run later
        if run == True:
          self.tf(self.f)
          self._thermal_noise_base()
          self.compute_all_noise(self.f)
          self.shot_noise(self.f)
          self.tf(self.f)
          self.vp2p(self.f)

    def _init_params(self, kwargs):
        """Unified handler for all parameters with defaults."""
        defaults = {
            # Circuit Model
            'A': 1e-4, 'n': 1.6, 'Rs': 1, 'Rsh': 1e3,
            'Voc': 0.64, 'Jsc': 35e-3,
            'Lo': 1e-6, 'Co': 1e-6, 'Rc': 10,
            # Physical Model (PV manufacturing params)
            'Na': 1e16 * 1e6, 'Nd': 1e19 * 1e6, 'L': 300e-6,
            'er': 11.68, 'ni': 1e10 * 1e6
        }

        for key, default in defaults.items():
            val = kwargs.get(key, default)
            setattr(self, key, to_scal_Nx1(self.no_pv, val))

    def _scale_params(self):
        """Scales area-dependent parameters."""
        scale = self.A * 1e4
        self.Rsh = self.Rsh/ scale
        self.Rs = self.Rs/ scale
        self.Isc = self.Jsc * scale

    def _calc_dc_bias(self):
        """Sets up Iph, Voc, I0 based on Irradiance."""
        if not hasattr(self, 'Isc'): self.Isc = self.Jsc

        self.Iph = self.Isc.copy()
        self.Vt = Constants.kB * Constants.T / Constants.q

        # Scale for Irradiance
        self.Iph *= (self.Gamb+self.Gsignal) / self.Gref
        self.Isc = self.Iph # Approximation for Isc at new G
        self.Rsh *= self.Gref / (self.Gamb + self.Gsignal)
        self.Voc += self.Vt * np.log((self.Gamb + self.Gsignal) / self.Gref + 1e-20) # Avoid log(0)

        # Calculate Saturation Current I0
        num = self.Isc - self.Voc / self.Rsh
        den = np.exp(self.Voc / (self.n * self.Vt)) - 1
        self.I0 = num / den

    def pv_current(self, V):
        """
        Vectorized Analytic Solution using Lambert W function.
        """
        V = np.asarray(V)
        Rs, Rsh = self.Rs, self.Rsh
        Iph, I0 = self.Iph, self.I0
        nVt = self.n * self.Vt

        R_sum = Rs + Rsh

        # 1. Linear Term (Ohmic limit)
        term_linear = (Rsh * (Iph + I0) - V) / R_sum

        # 2. Lambert Argument (Theta)
        common_factor = Rsh / (nVt * R_sum)
        exponent_term = np.exp(common_factor * (V + Rs * (Iph + I0)))
        pre_factor = I0 * Rs * common_factor

        theta = pre_factor * exponent_term

        # 3. Solve
        w_val = lambertw(theta).real

        return term_linear - (nVt / Rs) * w_val

    def calc_capacitance(self):
        """Calculates junction capacitance using physics parameters."""
        es = self.er * Constants.eo
        q = Constants.q

        no = self.ni**2 / self.Na
        vbi = self.Vt * np.log(self.Na * self.Nd / self.ni**2)

        # Depletion Capacitance
        denom = 2 * (self.Na + self.Nd) * (vbi - self.V + 1e-6)
        denom[denom < 0] = 1e-20
        c_dep = self.A * np.sqrt((q * es * self.Na * self.Nd) / denom)

        # Diffusion Capacitance
        c_dif = self.A * q * self.L * no * np.exp(self.V / self.Vt) / self.Vt

        self.C = c_dep + c_dif

    def find_bw(self, verbose=False):
        """Calculates 3dB Bandwidth."""
        r_eq_inv = 1/self.Rsh + 1/self.r + 1/(self.Rs + self.Rc)
        self.req = 1 / r_eq_inv

        self.BW = 1 / (2 * np.pi * self.req * self.C)
        if verbose: print(f"BW: {self.BW}")

    def tf(self, f):
        """Calculates Transfer Function H_pv(f)."""
        # Dimensions: [N, 1, M] for freq, [N, V, 1] for components
        w = 2 * np.pi * f[:, None, :]
        r, C, Rl = self.r[..., None], self.C[..., None], self.Rl[..., None]

        # Impedances
        Zp = 1 / (1/self.Rsh[..., None] + 1/r + 1j*w*C)
        Zdc = 1j*w*self.Lo[..., None] + Rl
        Zac = self.Rc[..., None] + 1/(1j*w*self.Co[..., None])

        # Divider Chain
        Zout = 1 / (1/Zac + 1/Zdc) + self.Rs[..., None]
        h1 = Zp / (Zp + Zout)
        h2 = Zdc / (Zac + Zdc)

        self.hpv = np.abs(h1 * h2 * self.Rc[..., None])

    # --- Noise Methods ---
    def _thermal_noise_base(self):
        """Helper to init thermal noise densities."""
        kT = 4 * Constants.kB * Constants.T
        self.No_r = kT * self.r
        self.No_Rs = kT * self.Rs
        self.No_Rsh = kT * self.Rsh
        self.No_Rl = kT * self.Rl
        self.No_Rc = kT * self.Rc

    def compute_all_noise(self, f):
        """
        Computes all 5 thermal noise sources (Rc, Rs, Rl, Rsh, r)
        and integrates them to get total thermal noise.
        """
        # 1. Initialize Thermal Noise Densities (4kTR)
        self._thermal_noise_base()

        # 2. Setup Vectorized Variables
        # Shape: (N, 1, M) for freq, (N, V, 1) for components
        w = 2 * np.pi * f[:, None, :]
        r = self.r[..., None]
        C = self.C[..., None]
        Rl = self.Rl[..., None]
        Rs = self.Rs[..., None]
        Rsh = self.Rsh[..., None]
        Rc = self.Rc[..., None]

        # 3. Common Impedance Blocks
        Z_Co = 1 / (1j * w * self.Co[..., None])
        Z_Lo = 1j * w * self.Lo[..., None]
        Z_Comm = Rc + Z_Co          # Branch with Rc and Co
        Z_EH = Rl + Z_Lo            # Branch with Rl and Lo (Energy Harvester)

        # --- A. Noise from Rc (Contact Resistance) ---
        J_p = 1/r + 1j*w*C + 1/Rsh
        Z_source = Rs + (1/J_p)
        Z_sp = 1 / (1/Z_source + 1/Z_EH)
        den_rc = Z_Comm + Z_sp
        self.n_rc = np.abs(Rc / den_rc)**2 * self.No_Rc[..., None]

        # --- B. Noise from Rs (Series Resistance) ---
        r2s = 1 / (1/Z_EH + 1/Z_Comm)
        u1_rs = r2s / (Rs + (1/J_p) + r2s)
        u2_rs = Rc / Z_Comm
        self.n_rs = np.abs(u1_rs * u2_rs)**2 * self.No_Rs[..., None]

        # --- C. Noise from Rl (Load Resistance) ---
        r1l = (1/J_p) + Rs
        r2l = Z_Comm
        r3l = 1 / (1/r1l + 1/r2l)
        u1_rl = Rc / Z_Comm
        u2_rl = r3l / (Rl + r3l + Z_Lo)
        self.n_rl = np.abs(u1_rl * u2_rl)**2 * self.No_Rl[..., None]

        # --- D. Noise from Rsh (Shunt Resistance) ---
        # Helper impedances for parallel branches
        h1 = 1 / Z_Comm
        h2 = 1 / Z_EH
        z_load_eq = 1 / (h1 + h2) # Equivalent load impedance seen from Rs

        # Impedance looking back into the diode/capacitor junction
        denom_rsh = 1/(Rs + z_load_eq) + 1/r + 1j*w*C
        z_node_rsh = 1 / denom_rsh

        u1_rsh = z_node_rsh / (Rsh + z_node_rsh)
        u2_rsh = z_load_eq / (z_load_eq + Rs)
        u3_rsh = Rc / Z_Comm

        self.n_rsh = np.abs(u1_rsh * u2_rsh * u3_rsh)**2 * self.No_Rsh[..., None]

        # --- E. Noise from r (Dynamic Resistance) ---
        # Similar topology to Rsh, but 'r' is the source
        denom_r = 1/(Rs + z_load_eq) + 1/Rsh + 1j*w*C
        z_node_r = 1 / denom_r

        u1_r = z_node_r / (r + z_node_r)
        u2_r = z_load_eq / (z_load_eq + Rs)
        u3_r = Rc / Z_Comm

        self.n_r = np.abs(u1_r * u2_r * u3_r)**2 * self.No_r[..., None]

        # 4. Integration (Total Noise Current^2)
        # Integrate PSD over frequency (axis=2)
        self.int_rc = np.trapz(self.n_rc, f[:, None, :], axis=2)
        self.int_rs = np.trapz(self.n_rs, f[:, None, :], axis=2)
        self.int_rl = np.trapz(self.n_rl, f[:, None, :], axis=2)
        self.int_rsh = np.trapz(self.n_rsh, f[:, None, :], axis=2)
        self.int_r = np.trapz(self.n_r, f[:, None, :], axis=2)

        # 5. Total Thermal Noise Sum
        self.th_noise = (self.int_rc + self.int_rs + self.int_rl +
                         self.int_rsh + self.int_r)

    def shot_noise(self, f):
        """Calculates shot noise contribution."""
        # Transfer function magnitude squared
        t = np.abs(self.hpv)**2
        # Integration
        integral = np.trapz(t, f[:, None, :], axis=2)
        self.sh_noise = 2 * Constants.q * self.Iph * integral

    def vp2p(self,f):
      self.vac = self.hpv * self.iac[:,None]

In [77]:
class PhyNet:
  """
  Physics Network Simulation Kernel.

  This class acts as the central controller for the entire optical wireless simulation.
  It orchestrates the initialization of the environment, nodes, and physics engines,
  and executes the main simulation pipeline:
  
  1.  **Build Phase:** Constructs Room, Sensors (SN), Masters (MN), and Ambient nodes (AN).
  2.  **Physics Phase:** Calculates Noise, Geometric Gains, and received signal powers.
  3.  **Optimization Phase (Optional):** Calculates minimum required Tx power to meet BER targets.
  4.  **Analysis Phase:** Computes SNR, Link Margins, and Bit Error Rates (BER).

  Attributes:
      room_builder, sn, mn, an: Builder objects for components.
      snm, mnm, amn: Manager objects for component logic.
      ogains: The Optical Physics Engine (oPhyGains).
      snr_d, ber_d: Downlink performance metrics.
      snr_u, ber_u: Uplink performance metrics.
  """
  def __init__(self, design, budget_run = False):
    """
    Initialize the Simulation Kernel.

    Args:
        design (dict): The master configuration dictionary.
        budget_run (bool): If True, triggers a 'Link Budget' run which calculates 
                           the minimum required Tx power to hit target BERs 
                           before running the full metric analysis.
    """
    self.room_builder = RoomBuilder(design)
    self.room = Room(self.room_builder)
    self.sn = NodeBuilder(design, "sensors")
    self.mn = NodeBuilder(design, "masters")
    self.an = NodeBuilder(design, "ambient_nodes")
    self.snm = SNManager(self.sn)
    self.mnm = MNManager(self.mn)
    self.amn = ANManager(self.an)
    self.ogains = oPhyGains(self.room,self.mnm,self.snm,self.amn)

    #fetch PV circuit params if they exist 
    self.pv_kwargs = design.get("PV_circuit", {})  

    self.compute_noise()
    if budget_run == True:
      self.set_tx_power()
    self.ogains.compute_downlink()
    self.ogains.compute_uplink()
    self.compute_metrics()


  def calc_min_ow_tx_power(self,target_ber):
    """
    Calculates the minimum Optical Wireless (OW) transmit power required to achieve a target BER.

    Uses the Inverse Q-function ($Q^{-1}$) to determine the required SNR, then solves 
    the link budget equation in reverse:
    
    $$P_{tx, req} = \\frac{2 \cdot Q^{-1}(BER) \cdot \sqrt{\\sigma_{noise}^2}}{H \cdot c_d}$$

    Args:
        target_ber (float): The desired Bit Error Rate (e.g., 3.8e-3).
    """
    self.target_ber = to_scal_Nx1(self.snm.ir_flag,target_ber)
    self.target_g = Qinv(self.target_ber)
    self.target_snr = self.target_g**2
    self.p_req_los = 2 * self.target_g * np.sqrt(self.x_u_noise) / (self.ogains.h_u_los * self.mnm.c_d)
    self.p_req_diff = 2 * self.target_g * np.sqrt(self.x_u_noise) / (self.ogains.h_u_diff * self.mnm.c_d)
    self.p_req_ris = 2 * self.target_g * np.sqrt(self.x_u_noise) / (self.ogains.h_u_ris * self.mnm.c_d)
    self.p_req_total = 2 * self.target_g * np.sqrt(self.x_u_noise) / ((self.ogains.h_u_ris + self.ogains.h_u_los + self.ogains.h_u_diff) * self.mnm.c_d)

    self.p_req = np.min(self.p_req_total,axis = 1).reshape(-1,1) # minimum power required for the best link
    self.p_sel = np.argmin(self.p_req_total,axis = 1, keepdims = True) #find with which MN the best link occurs

  def calc_min_rf_tx_power(self):
    """
    Calculates the minimum RF transmit power to exceed receiver sensitivity.
    
    $$P_{tx} \\ge Sensitivity + PathLoss_{dB}$$
    """
    self.p_rf_x = self.mnm.sensitivity.T + self.ogains.h_u_rf
    self.p_rf = np.min(self.p_rf_x, axis = 1).reshape(-1,1) # minimum power required for the best link
    self.p_rf_sel = np.argmin(self.p_rf_x, axis = 1, keepdims = True) #find with which MN the best link occurs

  def set_tx_power(self,target_ber = 3.8e-3):
    """"
    Optimizes Transmit Power for Uplinks.
    
    Overwrites the default Tx power settings for both IR and RF uplinks based on
    the minimum power needed to meet communication requirements (Target BER for Optical,
    Sensitivity limit for RF).
    """
    if self.snm.rf_flag > 0:
        self.calc_min_rf_tx_power()
        self.snm.RFTx_elements.p = self.p_rf
        
    if self.snm.ir_flag > 0:  
        self.calc_min_ow_tx_power(target_ber)
        self.snm.OTx_elements.p = self.p_req
    



  def compute_noise(self):
    """
    Computes total noise power/irradiance for all link types.
    
    **1. Downlink (PV Receivers):**
    Calculates noise **Irradiance** ($W/m^2$).
    * Sum of Artificial Light (Lamps) + Natural Light (Windows).
    
    **2. Downlink (Photodiode Receivers):**
    Calculates noise **Current Variance** ($A^2$).
    * Thermal Noise (from TIA model).
    * Shot Noise (from background light: Lamps + Windows).
    
    **3. Uplink (Master Receivers):**
    Calculates noise **Current Variance** ($A^2$).
    * Thermal + Shot Noise at the Base Station receiver.
    """
    #downlink noise for the PV-based Rxs
    self.g_d_noise = None
    self.flag_pv = (self.snm.ORx_elements.type_Rx == 1).flatten()
    self.no_pv = np.sum(self.flag_pv)
    if self.flag_pv.any():
      self.g_d_noise = np.zeros((1,self.no_pv))
      if self.room.Tx_windows_elements is not None:
        self.gix_d_noise = self.ogains.ix_d_noise[:,self.flag_pv]/self.snm.ORx_elements.A.flatten()[self.flag_pv]
        self.g_d_noise = self.gix_d_noise
      if self.amn.OTx_elements is not None:
        self.gis_d_noise =  self.ogains.is_d_noise[:,self.flag_pv]/self.snm.ORx_elements.A.flatten()[self.flag_pv]
        self.g_d_noise = self.g_d_noise + self.gis_d_noise
    

    #for the downlink we calculate the required BW - we consider all MN to transmit at the same bit rate
    self.Rb_d = to_scal_Nx1(self.snm.no_sensors,self.mnm.Rb_down)
    self.n_sp_d = to_scal_Nx1(self.snm.no_sensors,self.mnm.n_sp_d)    
    self.BW_d = self.Rb_d/self.n_sp_d
     
        
      
    #downlink noise for the PD-based Rxs
    self.x_d_noise = None
    self.flag_pd = (self.snm.ORx_elements.type_Rx == 0).flatten()
    if self.flag_pd.any():
      self.tia_noise_downlink = self.snm.tia.calc_noise_power(self.BW_d.reshape(-1,))[self.flag_pd]
      self.x_d_noise = self.snm.tia.calc_noise_power(self.BW_d.reshape(-1,))[self.flag_pd]
      if self.room.Tx_windows_elements is not None:
        self.xis_d_noise = 2 * Constants.q * self.ogains.is_d_noise[:,self.flag_pd] * self.BW_d.reshape(-1,)[self.flag_pd]
        self.x_d_noise = self.tia_noise_downlink + self.xis_d_noise
      if self.amn.OTx_elements is not None:
        self.xix_d_noise = 2 * Constants.q * self.ogains.ix_d_noise[:,self.flag_pd] * self.BW_d.reshape(-1,)[self.flag_pd]
        self.x_d_noise = self.x_d_noise + self.xix_d_noise

    #uplink noise for the PD-based Rxs
    self.x_u_noise = None
    if self.snm.ir_flag > 0:
      self.Rb_u = to_scal_Nx1(self.snm.ir_flag,self.snm.Rb_up)
        
      self.n_sp_u = to_scal_Nx1(self.snm.ir_flag,self.snm.n_sp_u)   
      self.BW_u = self.Rb_u/self.n_sp_u  
      self.tia_noise_uplink = self.mnm.tia.calc_noise_power(self.BW_u.reshape(-1,))
      self.x_u_noise = self.mnm.tia.calc_noise_power(self.BW_u.reshape(-1,))
      if self.room.Tx_windows_elements is not None:
        self.xis_u_noise = 2 * Constants.q * self.ogains.is_u_noise * self.BW_u.reshape(-1,1)
        self.x_u_noise = self.tia_noise_uplink.reshape(-1,1) + self.xis_u_noise
      if self.amn.OTx_elements is not None:
        self.xix_u_noise = 2 * Constants.q * self.ogains.ix_u_noise * self.BW_u.reshape(-1,1)
        self.x_u_noise = self.x_u_noise + self.xix_u_noise



  def compute_metrics(self):
    """
    Computes final performance metrics (SNR, BER, Link Margin).

    **Methodology:**
    1.  **PV Receivers:**
        * Instantiates a `PV` circuit model for each receiver.
        * Inputs calculated Signal and Noise Irradiance ($W/m^2$).
        * Solves for Maximum Power Point (MPP).
        * Calculates SNR based on AC small-signal power vs. Thermal/Shot noise power.
    
    2.  **PD Receivers:**
        * Standard SNR formulation: $SNR = \\frac{I_{sig}^2}{4 \\sigma_{noise}^2}$.
        * BER via Q-function: $BER = Q(\\sqrt{SNR})$.
    
    3.  **Hybrid Uplink (RF + Optical):**
        * **RF:** Calculates Link Margin ($P_{rx} - Sensitivity$).
        * **Optical:** Calculates SNR and BER.
        * Selects the best performing link (Max SNR / Max Margin).
    """
    #calculate metrics for downlink

    self.snr_d = np.zeros(self.snm.no_sensors)
    self.snr_d_dB = np.zeros(self.snm.no_sensors)

    #calculate for PV-based Rxs
    if self.flag_pv.any():
      self.g_d_los = self.ogains.i_d_los[:,self.flag_pv]/self.snm.ORx_elements.A.flatten()[self.flag_pv]
      self.g_d_diff = self.ogains.i_d_diff[:,self.flag_pv]/self.snm.ORx_elements.A.flatten()[self.flag_pv]
      self.g_d_ris = self.ogains.i_d_ris[:,self.flag_pv]/self.snm.ORx_elements.A.flatten()[self.flag_pv]


      self.g_d_signal = self.g_d_los + self.g_d_diff + self.g_d_ris

      #let's consider in this case all MN LEDs transmit the same data simultaneously
      self.g_d_signal_total = np.max(self.g_d_signal, axis = 0)

      self.pvx = PV(Gsignal = self.g_d_signal_total, Gamb = self.g_d_noise, 
                    A = self.snm.ORx_elements.A.flatten()[self.flag_pv] ,
                    unscaled = True, run = True, **self.pv_kwargs)

      self.signal_pv = np.take_along_axis(self.pvx.vac[..., -1], self.pvx.ind.reshape(-1,1), axis=1)/0.707


      self.noise_pv = 4 * (np.take_along_axis(self.pvx.th_noise, self.pvx.ind.reshape(-1,1), axis=1) + np.take_along_axis(self.pvx.sh_noise, self.pvx.ind.reshape(-1,1), axis=1))
      self.snr_pv = self.signal_pv**2 / self.noise_pv
      self.snr_pv_dB = 10 * np.log10(self.snr_pv)

      self.snr_d[self.flag_pv] = self.snr_pv.reshape(-1,)
      self.snr_d_dB[self.flag_pv] = self.snr_pv_dB.reshape(-1,)

    #for PD-based Rxs

    if self.flag_pd.any():
      self.x_d_los = self.ogains.i_d_los[:,self.flag_pd]
      self.x_d_diff = self.ogains.i_d_diff[:,self.flag_pd]
      self.x_d_ris = self.ogains.i_d_ris[:,self.flag_pd]

      self.x_d_signal = self.x_d_los + self.x_d_diff + self.x_d_ris

      self.x_d_signal_total = np.sum(self.x_d_signal, axis = 0)


      self.snr_pd = self.x_d_signal_total**2 / (4 * self.x_d_noise)
      self.snr_pd_dB = 10 * np.log10(self.snr_pd)

      self.snr_d[self.flag_pd] = self.snr_pd.reshape(-1,)
      self.snr_d_dB[self.flag_pd] = self.snr_pd_dB.reshape(-1,)


    self.BER_d = Qfunction(np.sqrt(self.snr_d))


    if self.snm.rf_flag > 0:

      #uplink - RF first
      self.hrf = self.ogains.h_u_rf
      #find incident RF power
      self.p_u_rf_m = self.snm.RFTx_elements.p - self.hrf

      #find power margin
      self.rf_margin = self.p_u_rf_m - self.mnm.sensitivity.T

      #find the best link

      self.rf_best_margin = np.max(self.rf_margin, axis = 1)
      self.u_sel_rf = np.argmax(self.rf_margin, axis = 1, keepdims=True)

      #find incident power for best margin
      self.p_u_rf = np.take_along_axis(self.p_u_rf_m, self.u_sel_rf, axis = 1)


       #optical wireless uplinks
      self.snr_u = self.ogains.i_u_signal**2/(4*self.x_u_noise)
      self.snr_u_dB = 10 * np.log10(self.snr_u)
      self.BER_u = Qfunction(np.sqrt(self.snr_u))

       #select best link
      self.snr_u_sel = np.max(self.snr_u,axis = 1)
      self.u_sel_ow = np.argmax(self.snr_u,axis = 1, keepdims = True)
      self.ber_u_sel = np.take_along_axis(self.BER_u, self.u_sel_ow, axis = 1)

In [78]:
class IRdriver:
    """
    Models the electro-optical characteristics of an Infrared LED driver.
    
    Robustly handles initialization via a configuration dictionary.
    """

    def __init__(self, **kwargs):
        """
        Initialize the IR Driver model using keyword arguments.
        
        Args:
            **kwargs: Dictionary containing:
                - imax (float): Saturation current limit (default: 100mA).
                - imin (float): Cut-off current limit (default: 0mA).
                - pol (array): Coefficients for I -> P (default provided).
                - polinv (array): Coefficients for P -> I (default provided).
        """
        # 1. Extract params with defaults using kwargs.get()
        self.imax = kwargs.get('imax', 100e-3)
        self.imin = kwargs.get('imin', 0e-3)
        
        # Default polynomials (Polynomials for standard IR LED)
        default_pol = np.array([ 1.35376064e-01,  1.86846949e-01, -1.01789073e-04])
        default_polinv = np.array([-1.74039667e+01, 5.32917840e+00, 5.61867428e-04])
        
        # Ensure inputs are numpy arrays
        self.pol = np.array(kwargs.get('pol', default_pol))
        self.polinv = np.array(kwargs.get('polinv', default_polinv))

        # 2. Pre-calculate limits
        self.Pmax = np.polyval(self.pol, self.imax)
        self.Pmin = np.polyval(self.pol, self.imin)

    def calc_I(self, P):
        """Calculates Drive Current from Optical Power."""
        I = np.polyval(self.polinv, P)
        
        # Vectorized Saturation Checks
        I = np.atleast_1d(I)
        P = np.atleast_1d(P)
        
        I[I >= self.imax] = np.inf
        I[P >= self.Pmax] = np.inf
        
        if I.size == 1: return I.item()
        return I

    def calc_P(self, I):
        """Calculates Optical Power from Drive Current."""
        return np.polyval(self.pol, I)

In [145]:
def RF_calc_I(P, **kwargs):
    """
    Estimates the power consumption current of the RF transmitter.
    
    Args:
        P (float or np.ndarray): RF Transmit Power (dBm).
        **kwargs: 
            - p_min (float): Min power limit. Default: -20
            - p_max (float): Max power limit. Default: 5
            - pol (array): [slope, intercept]. Default: [0.24, 8.8]
    """
    # 1. Extract parameters from kwargs with fallbacks
    p_min = kwargs.get('p_min', -20)
    p_max = kwargs.get('p_max', 5)
    pol = kwargs.get('pol', np.array([0.24, 8.8]))

    # 2. Ensure P is a numpy array for consistent masking/clipping
    P_bounded = np.array(P, copy=True)
    
    # 3. Apply the clipping/logic
    # Anything below p_min is treated as p_min
    P_bounded[P_bounded < p_min] = p_min
    
    # Identify values exceeding p_max for infinity current later
    over_limit_mask = P_bounded > p_max

    # 4. Calculation using the bounded power
    # np.atleast_1d handles the "Node 0" scalar case to prevent size mismatches
    I = np.atleast_1d(np.polyval(pol, P_bounded) * 1e-3)
    
    # 5. Apply infinity to current for values exceeding limits
    if np.any(over_limit_mask):
        I = I.astype(float) # Ensure we can hold np.inf
        I[over_limit_mask] = np.inf
        
    # Return as scalar if input was scalar, else return array
    return I.item() if np.isscalar(P) else I

In [146]:
def to_vec_Nx3(N,val):
    """
    Broadcasting helper: Ensures the input 'val' is shaped as (N, 3).
    
    Useful for expanding a single position/normal vector to match a batch of N elements.
    
    Args:
        N (int): Target number of rows.
        val (array-like): Input vector(s). Can be (3,), (1, 3), or (N, 3).
        
    Returns:
        np.ndarray: Array of shape (N, 3).
    """
    arr = np.array(val)
    if arr.ndim == 1:
    # e.g., [0,0,1] -> (1,3) -> tile to (N,3)
        arr = arr.reshape(1, 3)
    if arr.shape[0] == 1 and N > 1:
        arr = np.tile(arr, (N, 1))
    return arr

    # --- Helper for Scalar Fields (N, 1) ---
def to_scal_Nx1(N,val, default_val=0):
    """
    Broadcasting helper: Ensures the input 'val' is shaped as (N, 1).
    
    Useful for expanding a scalar property (like Power or Area) to match a batch of N elements.
    
    Args:
        N (int): Target number of rows.
        val (scalar or array-like): Input value.
        default_val (float): Value to use if 'val' is None.
        
    Returns:
        np.ndarray: Array of shape (N, 1).
    """
    if val is None:
        val = default_val

    if is_scalar(val):
        return np.full((N, 1), val)

    arr = np.array(val)
    if arr.ndim == 1:
        return arr.reshape(N, 1)
    return arr

def normalize_bool_array(x, N):
    """
    Converts a boolean scalar or list into a boolean numpy array of size N.
    """
    if isinstance(x, (bool, np.bool_)):
        return np.full(N, x, dtype=bool)
    return np.asarray(x, dtype=bool)

def as_array_of_size(x, N):
    """
    Enforces that input 'x' becomes an array of size N.
    Raises ValueError if dimensions mismatch.
    """
    if np.isscalar(x):
        return np.full(N, x)
    arr = np.asarray(x)
    if arr.size == 1:
        return np.full(N, arr.item())
    
    if arr.size != N:
        raise ValueError(f"Expected size {N}, got {arr.size}")
    return arr

def solar_panel_angular_efficiency(cos_inc: np.ndarray) -> np.ndarray:
    """
    Models the angular loss of a Solar Panel (deviations from Lambertian).
    
    Calculates efficiency degradation based on the incidence angle (theta) using 
    a 5th-order polynomial fit derived from experimental data.
    
    Args:
        cos_inc (np.ndarray): Cosine of the incidence angle.
        
    Returns:
        np.ndarray: Efficiency scaling factor (0.0 to ~1.0).
    """
    p_p = np.array([-1.81907071e-09,  3.00750020e-07, -1.82841164e-05,  4.57546496e-04,
                    -4.11754977e-03,  1.00666212e+00])
    p_s = np.poly1d(p_p)
    theta = np.rad2deg(np.arccos(np.clip(cos_inc, -1.0, 1.0)))
    efficiency = p_s(theta)
    efficiency[theta >= 90] = 0
    return np.maximum(0, efficiency)

def is_scalar(x):
    """
    Checks if a value is effectively a scalar (int, float, or 0-d array).
    """
    if x is None:
        return False
    return np.isscalar(x) or (isinstance(x, np.ndarray) and x.ndim == 0)

def Qfunction(x):
    """
    Standard Gaussian Q-function.
    Q(x) = 0.5 * erfc(x / sqrt(2))
    """
    return 0.5 * erfc( x/np.sqrt(2) )

def Qinv(y):
    """
    Inverse Gaussian Q-function.
    """
    return np.sqrt(2) * erfcinv( 2 * y )

In [147]:
class TIA:
    """
    Models the frequency response and noise performance of a Transimpedance Amplifier (TIA).
    
    Attributes:
        RF (float or np.array): Feedback resistance (Ohms).
        Vn (float or np.array): Op-amp input voltage noise density (V/sqrt(Hz)).
        In (float or np.array): Op-amp input current noise density (A/sqrt(Hz)).
        fncV (float or np.array): 1/f corner frequency for voltage noise (Hz).
        fncI (float or np.array): 1/f corner frequency for current noise (Hz).
        temperature (float or np.array): Temperature in Kelvin.
    """

    def __init__(self, **argv):
        self.RF = argv.pop('RF')
        self.Vn = argv.pop('Vn')
        self.In = argv.pop('In')
        self.fncV = argv.pop('fncV')
        self.fncI = argv.pop('fncI')
        self.temperature = argv.pop('temperature')

    # -----------------------------
    # Vectorized transfer function
    # -----------------------------
    def CF(self, B):
        return 1 / (2 * np.pi * B * self.RF)   # (NB,)

    def ZF(self, f, B):
        """
        f: (Nf,)
        B: (NB,)
        return: (NB, Nf)
        """
        f = f[None, :]
        B = B[:, None]
        CF = self.CF(B)

        return self.RF / (1 + 1j * 2 * np.pi * f * CF * self.RF)

    # -----------------------------
    # PSDs (all vectorized)
    # -----------------------------
    def RF_psd(self, f, B):
        return (4 * Constants.kB * self.temperature / self.RF) * np.ones((B.size, f.size))

    def SV_psd(self, f, B):
        f = f[None, :]
        Z = self.ZF(f.squeeze(), B)
        return (self.Vn**2 + self.Vn**2 * self.fncV / f) / np.abs(Z)**2

    def SI_psd(self, f, B):
        f = f[None, :]
        return self.In**2 + self.In**2 * self.fncI / f

    def psd(self, f, B):
        return (
            self.RF_psd(f, B)
            + self.SV_psd(f, B)
            + self.SI_psd(f, B)
        )

    # -----------------------------
    # Noise power (vectorized)
    # -----------------------------
    def calc_noise_power(self, B, Nf=1000, fmin=0.1):
        B = np.atleast_1d(B)

        # Normalized frequency grid (0..1), shared
        x = np.linspace(0, 1, Nf)
        f = fmin + x * (B[:, None] - fmin)  # (NB, Nf)

        psd_vals = self.psd(f[0], B)        # f[0] gives base grid
        psd_vals *= (f <= B[:, None])       # enforce per-B cutoff

        return np.trapz(psd_vals, f, axis=1)

In [148]:
ph_net = PhyNet(DUMMY_DESIGN,True)



Adding 'ris_west'. Removed 24 tiles.
Adding 'ris_east'. Removed 24 tiles.
Adding 'win_north'. Removed 48 tiles.


In [149]:
ph_net.snr_u

array([[7.1253872 , 3.57243126],
       [5.0911559 , 7.1253872 ],
       [7.1253872 , 4.65544922]])

In [150]:
ph_net.snr_u_dB

array([[8.52808469, 5.52963881],
       [7.06816396, 8.52808469],
       [8.52808469, 6.67961594]])

In [151]:
ph_net.ogains.i_u_signal

array([[7.68126947e-08, 6.09668650e-08],
       [6.49287384e-08, 8.61025786e-08],
       [7.68126947e-08, 6.95973526e-08]])

In [152]:
class EnergyManager:
    """
    Energy Consumption & Harvesting Manager.

    This class simulates the power profile of sensor nodes over a defined operation cycle.
    It integrates hardware specifications, task computational loads, and communication 
    overhead to compute the total energy drain per cycle. It also calculates the 
    energy harvesting potential for PV-equipped nodes.

    **Energy Model:**
    The cycle is decomposed into discrete states:
    1.  **Initialization:** Wake-up and boot sequence.
    2.  **Sensing:** ADC sampling and sensor readout.
    3.  **Processing:** MCU computation (compression, formatting).
    4.  **Uplink (Tx):** Data transmission (IR LED or RF Radio).
    5.  **Turnaround (Wait):** Waiting for Downlink (ACK/Data).
    6.  **Downlink (Rx):** Receiving data.
    7.  **Sleep:** Low-power state for the remainder of the cycle period ($T_{cycle}$).

    Attributes:
        f_mcu, f_s (np.ndarray): Clock frequencies for MCU and Sampling [Hz].
        V (np.ndarray): Supply Voltage [V].
        I_* (np.ndarray): Current consumption for various hardware states [A].
        N_* (np.ndarray): Number of clock cycles for tasks.
        L_* (np.ndarray): Data packet lengths [bits].
        br_* (np.ndarray): Bit rates for communication [bps].
        harvesting_hours (np.ndarray): Effective hours of light exposure per day for PV nodes.
        batt_charge (np.ndarray): Total battery energy capacity in Joules.
    """
    def __init__(self, phy_net, design):
        """
        Initialize the Energy Manager.

        Parses the configuration dictionary to populate hardware and protocol parameters.
        Uses a hierarchical lookup (Design Profile -> Protocol -> Defaults) to ensure
        all parameters are set, broadcasting scalars to arrays of size $N$ (number of sensors).
        
        It also initializes the battery state and assigns harvesting hours specifically 
        to nodes identified as having PV capabilities.

        Args:
            phy_net (PhyNet): The main simulation object (access to physics models).
            design (dict): Configuration dictionary containing 'nodes', 'energy_profile', etc.
        """
        self.pn = phy_net
        self.N = self.pn.snm.no_sensors

        self.nodes = design['nodes']['sensors']
        self.u_prof = design.get('energy_profile', {})
        self.u_prot = design.get('protocol', {})

        # 1. Hardware Specs
        self.f_mcu = self._v('f_mcu', EnergyDefaults.hardware, 'hardware')
        self.f_s   = self._v('f_s',   EnergyDefaults.hardware, 'hardware')
        self.V     = self._v('voltage', EnergyDefaults.hardware, 'hardware')
        self.I_mcu, self.I_adc = self._v('I_mcu', EnergyDefaults.hardware), self._v('I_adc', EnergyDefaults.hardware)
        self.I_act, self.I_ext = self._v('I_act', EnergyDefaults.hardware), self._v('I_ext', EnergyDefaults.hardware)
        self.I_sleep, self.I_wake = self._v('I_sleep', EnergyDefaults.hardware), self._v('I_wake', EnergyDefaults.hardware)

        # 2. Task Loads
        self.N_s_up = self._v('N_s_up', EnergyDefaults.tasks, 'tasks')
        self.N_c_up = self._v('N_c_up', EnergyDefaults.tasks, 'tasks')
        self.L_up   = self._v('L_up_bits', EnergyDefaults.tasks, 'tasks')
        print(self.L_up)
        self.L_dw   = self._v('L_dw_bits', EnergyDefaults.tasks, 'tasks')
        print(self.L_dw)
        # 3. Communication Overheads
        self.Rb_up = self._v('Rb_up', EnergyDefaults.communication, "communication") #-- νατο αλλαξω στα defaults
        self.Rb_down = self._v('Rb_down', EnergyDefaults.communication, "communication")
        print(self.Rb_up)
        print(self.Rb_down)
        self.t_init, self.t_wait = self._v('t_init', EnergyDefaults.communication), self._v('t_wait', EnergyDefaults.communication)
        self.T_cycle = self._v('T_cycle', EnergyDefaults.communication)
        self.ir_driver = IRdriver()    # να βαλω τα kwargs για driver  

        #4. Battery 
        self.batt_capacity_mAh = self._v('battery_capacity_mAh', EnergyDefaults.battery, "battery")
        self.V_batt = self._v('V_batt', EnergyDefaults.battery, "battery")
        self.initial_soc = self._v("initial_soc",  EnergyDefaults.battery, "battery")
        self.batt_charge = self.batt_capacity_mAh * self.V_batt * 3.6 * self.initial_soc #mAh to joules

        # 1. Start with 0 for everyone
        self.harvesting_hours = np.zeros(self.N)

        hh_input = self.u_prot.get('harvesting_hours', EnergyDefaults.communication['harvesting_hours'])
        
        # 3. Apply ONLY to PV nodes
        if hasattr(self.pn, 'flag_pv') and np.any(self.pn.flag_pv):
            try:
                # This automatically handles Scalar (broadcasts) OR Array (if size matches mask)
                self.harvesting_hours[self.pn.flag_pv] = hh_input
            except ValueError:
                # If sizes don't match, warn and fail safe to 0
                print(f"Warning: 'harvesting_hours' array size does not match number of PV nodes. Using 0.")
        
        # Placeholders for daily stats
        self.E_day_consumed = np.zeros(self.N)
        self.E_day_harvested = np.zeros(self.N)
        self.E_day_net = np.zeros(self.N)
        self.days_to_empty = np.zeros(self.N)



    def _v(self, key, default_dict, profile_sub=None):
        """Standardizes lookup: energy_profile[sub] -> protocol -> default_dict."""
        # 1. Search energy_profile sub-categories
        if profile_sub:
            d = self.u_prof.get(profile_sub, {})
            if key in d: return as_array_of_size(d[key], self.N)
        
        # 2. Search root protocol/profile and then defaults
        val = self.u_prof.get(key, self.u_prot.get(key, default_dict.get(key)))
        return as_array_of_size(val, self.N)

    
    def calc_cycle_energy(self):
        """
        Calculates the total energy consumption per operation cycle.

        **Physics:**
        1.  **Duration ($t$):** Calculated based on task size ($N_{cycles}$) and frequency ($f$), 
            or packet size ($L$) and bit rate ($R$).
            $$t_{proc} = N_{cycles} / f_{clk}, \quad t_{tx} = L_{bits} / R_{bps}$$
        
        2.  **Energy ($E$):** Integrated over time for all states.
            $$E_{active} = V \cdot \sum (I_{state} \cdot t_{state})$$
            $$E_{sleep} = V \cdot I_{sleep} \cdot (T_{cycle} - t_{active})$$

        **Note:** During the 'Wait' phase (turnaround time), the node is assumed to be in 
        an Active/Idle state (`I_act`), not sleep mode, to quickly respond to downlink.

        Returns:
            None (updates self.E_cycle in place).
        """
        
        # --- DURATIONS (s) ---
        self.d_init   = self.t_init
        self.d_sens_u = self.N_s_up / self.f_s
        self.d_proc_u = self.N_c_up / self.f_mcu
        
        # Uplink Transmission Time
        self.ir_m, self.rf_m = (self.nodes['uplink_type'] == 0), (self.nodes['uplink_type'] == 1)
        self.d_tx = np.zeros(self.N)
        self.d_tx[self.ir_m] = self.L_up[self.ir_m] / self.Rb_up[self.ir_m]
        self.d_tx[self.rf_m] = self.L_up[self.rf_m] / self.Rb_up[self.rf_m]

        # Downlink Reception Time
        self.d_wait   = self.t_wait #turnaround time
        self.d_rx     = self.L_dw / self.Rb_down

        # --- CURRENTS (A) ---
        self.I_sens = (self.I_adc + self.I_mcu + self.I_ext) 
        self.I_proc = (self.I_mcu) 
        self.I_rx   = (self.I_act + self.I_adc + self.I_mcu) 

        self.I_tx = np.zeros(self.N)
        if np.any(self.ir_m):
            self.I_tx[self.ir_m] = (self.I_mcu[self.ir_m] + 0.5 * self.ir_driver.calc_I(self.pn.snm.OTx_elements.p.reshape(-1,))) 
        if np.any(self.rf_m):
            #clip values to transceiver limits
            self.pn.snm.RFTx_elements.p[self.pn.snm.RFTx_elements.p < - 20] = -20
            self.pn.snm.RFTx_elements.p[self.pn.snm.RFTx_elements.p > 5] = np.inf
            self.I_tx[self.rf_m] = RF_calc_I(self.pn.snm.RFTx_elements.p.reshape(-1,)) #consumption from RF Transceiver during this phase usually already includes all relevant current

        # --- ENERGY INTEGRATION (J) ---
        
        self.E_active = self.V * (
            (self.I_wake * self.d_init) + #WU
            (self.I_sens * self.d_sens_u) + #SENS
             (self.I_proc * self.d_proc_u) + #PROC
              (self.I_tx * self.d_tx) + # UL
            (self.I_act * self.d_wait) + # Wait
            (self.I_rx * self.d_rx) # DL
        )

        
        self.d_total = self.d_init + self.d_sens_u + self.d_proc_u + self.d_tx + self.d_wait + self.d_rx 
        self.E_sleep = self.V * (self.I_sleep) * np.maximum(0, self.T_cycle - self.d_total)

        self.E_cycle = self.E_active + self.E_sleep

    def calc_harv_energy(self,mpp_eff):
      """
      Calculates the harvestable power for PV-equipped nodes.

      Extracts the Maximum Power Point (MPP) voltage and current from the `PV` 
      physics model (calculated in `PhyNet`) and applies an efficiency factor 
      for the DC-DC converter / PMIC.

      Args:
          mpp_eff (float): Efficiency of the Maximum Power Point Tracking (MPPT) circuit (0.0 - 1.0).
      
      Sets:
          self.p_raw (np.ndarray): Raw power at MPP [Watts].
          self.p_harv (np.ndarray): Extractable power after conversion losses [Watts].
      """
      self.p_raw = np.zeros(self.N)
      self.p_harv = np.zeros(self.N)
      
      if self.pn.flag_pv.any():
        v = np.take_along_axis(self.pn.pvx.V, self.pn.pvx.ind.reshape(-1,1),axis = 1)
        i = np.take_along_axis(self.pn.pvx.I, self.pn.pvx.ind.reshape(-1,1),axis = 1)
        self.p_r = v*i
        self.p_h = v*i * mpp_eff
        self.p_raw[self.pn.flag_pv] = self.p_r.flatten()
        self.p_harv[self.pn.flag_pv] = self.p_h.flatten()        
      else:
        print("There are not any PV-based receivers.")

    def calc_battery_life(self):
        """
        Calculates daily energy budget and estimates battery lifetime.

        Scales the single-cycle energy consumption to a full 24-hour day and compares it
        against the daily harvested energy to determine net energy flow.
        
        **Formula:**
        $$E_{day, cons} = E_{cycle} \times (24h / T_{cycle})$$
        $$E_{day, net} = (P_{harv} \times t_{sun}) - E_{day, cons}$$
        $$Days_{left} = E_{battery} / |E_{day, net}| \quad (\text{if } E_{day, net} < 0)$$

        Prints a tabular report of the results.
        """
        if not hasattr(self, 'E_cycle'):
            print("Running cycle energy calc first...")
            self.calc_cycle_energy()
        self.current_energy = self.batt_charge
            
        self.cycles_per_hour = 3600/self.T_cycle
        self.E_day_consumed = self.E_cycle * self.cycles_per_hour * 24.0

        if not hasattr(self, 'p_harv'):
            self.p_harv = np.zeros(self.N)
        
        # Only PV nodes have non-zero p_ext AND non-zero harvesting_hours
        self.E_day_harvested = self.p_harv * self.harvesting_hours * 3600.0
        self.E_day_net = self.E_day_harvested - self.E_day_consumed

        drain_mask = self.E_day_net < 0
        self.days_to_empty = np.full(self.N, np.inf)

        if np.any(drain_mask):
            loss_rate = np.abs(self.E_day_net[drain_mask])
            self.days_to_empty[drain_mask] = self.current_energy[drain_mask] / loss_rate

        print(f"{'Node ID':<10} {'Type':<10} {'Init(J)':<10} {'Cons/Day(J)':<15} {'Harv/Day(J)':<15} {'Net/Day(J)':<15} {'Life(Days)':<10}")
        print("-" * 90)
        for i in range(self.N):
             node_type = "PV + Batt " if self.pn.flag_pv[i] else "Battery"
             life_str = "Inf" if self.days_to_empty[i] == np.inf else f"{self.days_to_empty[i]:.2f}"
             
            
             print(f"{i:<10} {node_type:<10} {self.current_energy[i]:<10.1f} "
                   f"{self.E_day_consumed[i]:<15.2f} {self.E_day_harvested[i]:<15.2f} "
                   f"{self.E_day_net[i]:<15.2f} {life_str:<10}")

In [153]:
em = EnergyManager(ph_net,DUMMY_DESIGN)
em.calc_cycle_energy()
em.calc_harv_energy(0.8)
em.calc_battery_life()

[1024 8192  512 2048]
[64 64 64 64]
[100000. 100000. 100000. 100000.]
[15000. 15000. 15000. 15000.]
Node ID    Type       Init(J)    Cons/Day(J)     Harv/Day(J)     Net/Day(J)      Life(Days)
------------------------------------------------------------------------------------------
0          PV + Batt  6480.0     72.14           4.94            -67.20          96.43     
1          Battery    6480.0     195.75          0.00            -195.75         33.10     
2          PV + Batt  6480.0     55.70           9.98            -45.72          141.74    
3          Battery    6480.0     10.61           0.00            -10.61          610.67    


In [140]:
em.Rb_up

array([100000., 100000., 100000., 100000.])

In [141]:
s7 = {
    'meta': {
        'name': 'Hybrid_VLC_RF_IoT_Network_Equivalent',
        'description': 'Aligned with ph_net from test1(2) configuration.'
    },
    'environment': {
        'dimensions': np.array([5.0, 5.0, 3.0]),
        'wall_resolution': (20, 20),
        'reflectivity': {'floor': 0.3, 'ceiling': 0.8, 'walls': 0.5},
        'special_surfaces': [
            {'type': 'RIS', 'name': 'ris_west', 'center': [5, 2.5, 1.5], 'dims': [1.0, 1.0], 'const_axis': 0, 'normal': [1,0,0], 'resolution': (2,2), 'reflectivity': 0.9},
            {'type': 'RIS', 'name': 'ris_east', 'center': [0, 2.5, 1.5], 'dims': [1.0, 1.0], 'const_axis': 0, 'normal': [1,0,0], 'resolution': (2,2), 'reflectivity': 0.9},
            {'type': 'window', 'name': 'win_north', 'center': [2.5, 5, 1.5], 'dims': [2.0, 1.0], 'const_axis': 1, 'normal': [0,-1,0], 'resolution': (2,2), 'reflectivity': 0.1}
        ],
    },
    'nodes': {
        'masters': {
            'positions': np.array([[2.5, 2.5, 3.0], [3, 3, 3]]),
            'nT': [Constants.zm], # Wrapped in list to be 2D and avoid AxisError
            'nR': [Constants.zm], 
            'm': 1,
            'tx_power': 1,
            'rx_area': 1e-4,
            'IR_pass_filter': True,
            'sensitivity': -100
        },
        'sensors': {
            'positions': np.array([[1.0, 1.0, 0.85], [4.0, 4.0, 0.85], [2.5, 2.5, 0.85], [2, 2, 0]]),
            'rx_area': np.full(4, 1e-4),
            'nT': np.repeat(Constants.zp[None, :], 4, axis=0),
            'nR': np.repeat(Constants.zp[None, :], 4, axis=0),
            'rx_type': np.array([1, 0, 1, 0]), # Matches node types in test1(2) dummy
            'VLC_pass_filter': np.array([False, False, False, True]),
            'uplink_type': np.array([1, 1, 0, 1]), # Matches uplink types in test1(2) dummy
            "IR_tx_power": 0.015,
            "RF_tx_power": -20,
        },
        'ambient_nodes': {
            'positions': np.array([[3, 3, 3.0]]),
            'nT': [[0, 0, -1]],
            'tx_power': 2,
            'm': 1
        }
    },
    'TIA': {'RF': 1e6, 'CF': 1e-9, 'Vn': 15e-9, 'In': 400e-15, 'fncV': 1e3, 'fncI': 1e3, 'temperature': 300},
    
    # RENAME 'PV' to 'PV_circuit' to match ph_net.py lookup: design.get("PV_circuit", {})
    'PV_circuit': {'Rs': 1, 'Rsh': 1000, 'Jsc': 35e-3, 'Voc': 0.64, 'n': 1.6},

    'energy_profile': {
        'hardware': {
            'f_mcu': np.array([80e6, 80e6, 16e6, 48e6]),
            'f_s': np.array([1e3, 10e3, 1e3, 5e3]),
            'voltage': np.array([3.3, 3.3, 3.3, 1.8]),
            'I_mcu': np.array([4.0, 12.0, 4.0, 8.0])*1e-3,
            'I_adc': 2.0*1e-3,
            'I_act': 3.0*1e-3,
            'I_ext': np.array([0.5, 5.0, 1.0, 2.0])*1e-3,
            'I_sleep': 0.01*1e-3,
            'I_wake': 5.0*1e-3,
        },
        'tasks': {
            'N_samples_up': np.array([100, 5000, 100, 500]),
            'N_cycles_up': np.array([1e5, 5e6, 1e5, 8e5]),
            'L_up_bits': np.array([1024, 8192, 512, 2048]),
            'L_down_bits': 64,
        },
        'communication': {
            'n_sp': 0.4,
            'Rb_up': 4000,    # 4000 / 0.4 = 10,000 Hz BW (Equivalent to test1(2) default)
            'Rb_down': 4000,  # 4000 / 0.4 = 10,000 Hz BW
            't_init': 5e-3, 
            't_wait': 10e-3,
            'harvesting_hours': 5
        },
        'battery': {'battery_capacity_mAh': 500, 'initial_soc': 1, 'V_batt': 3.6}
    },
    'protocol': {'T_cycle': 6}
}

In [142]:
gf = PhyNet(s7,True)

Adding 'ris_west'. Removed 24 tiles.
Adding 'ris_east'. Removed 24 tiles.
Adding 'win_north'. Removed 48 tiles.


In [143]:
en = EnergyManager(gf,s7)

[1024 8192  512 2048]
[64 64 64 64]
[4000 4000 4000 4000]
[4000 4000 4000 4000]


In [144]:
en.calc_cycle_energy()
en.calc_harv_energy(0.8)
en.calc_battery_life()

Node ID    Type       Init(J)    Cons/Day(J)     Harv/Day(J)     Net/Day(J)      Life(Days)
------------------------------------------------------------------------------------------
0          PV + Batt  6480.0     99.86           4.94            -94.92          68.27     
1          Battery    6480.0     405.46          0.00            -405.46         15.98     
2          PV + Batt  6480.0     123.21          9.98            -113.22         57.23     
3          Battery    6480.0     64.48           0.00            -64.48          100.49    


In [132]:
en.pn.snm.RFTx_elements.p

array([[-42.95109722],
       [-43.96646813],
       [-42.82254617]])

In [136]:
en.pn.snm.RFTx_elements.p[ en.pn.snm.RFTx_elements.p < - 20] = -20

In [137]:
en.pn.snm.RFTx_elements.p

array([[-20.],
       [-20.],
       [-20.]])